# Ejercicio Práctico — Semana 3: Emociones en texto: NLP, clasificación y procesamiento distribuido

**Asignatura:** Inteligencia Artificial: Data Mining II — Maestría en Ciencias de Datos y Máquinas de Aprendizaje, mención IA (UIDE)
**Docente:** Miguel Angel Rosero Aguas
**Cubre:** Sesión 1 — Big Data con PySpark: Online Retail II  /  Sesión 2 — NLP, spaCy, LDA y Spark MLlib
**Grupo N.º:** 9

| # | Integrante | Rol en el equipo |
|---|---|---|
| 1 | Kevin Ambuludi | |
| 2 | Rodrigo Soria | |
| 3 | Christian Ruiz | |

## Índice

1. [Carga, limpieza y exploración del corpus](#fase-1)
2. [Análisis lingüístico con spaCy: POS y NER](#fase-2)
3. [Modelado de temas (LDA)](#fase-3)
4. [Clasificación clásica: Naive Bayes y SVM](#fase-4)
5. [Red neuronal recurrente (RNN)](#fase-5)
6. [Experimento: número de clases](#fase-6)
7. [Big Data: PySpark, MLlib y MapReduce](#fase-7)
8. [Desafío y evaluación de alternativas](#fase-8)
9. [Tendencias: BERTopic, LLM en el pipeline y Parquet](#fase-9)
10. [Fase final: aplicación al entorno profesional](#fase-final)

## Estructura del ejercicio

Este notebook va con la guía **Ejercicio_Practico_Semana3_Guia.docx**: el código y los gráficos se hacen aquí; en la guía responden por escrito
la **pregunta de cada fase** (la misma que aparece abajo de cada título) y la fase final individual.

| Bloque | Fases | Qué entregan |
|---|---|---|
| Datos y exploración | 1–3 | código + conclusión en el notebook; respuesta en la guía |
| Modelado y análisis | 4–7 | código + conclusión en el notebook; respuesta en la guía |
| Desafío y tendencias | 8–9 | evidencia en el notebook; respuesta en la guía |
| Aplicación profesional | final | una respuesta por integrante (notebook y guía) |

**Entrega en Canvas:** este notebook ejecutado y la guía completada (Word o PDF).

## Contexto

El área de experiencia del cliente de una empresa recibe miles de mensajes al día y quiere detectar automáticamente la emoción de cada uno para priorizar la atención. Antes de invertir, les pide un estudio con un corpus público de tweets etiquetados con 13 emociones: qué dice el texto (análisis lingüístico y temas), qué tan bien se puede clasificar la emoción según cuántas emociones se distingan, y si hace falta una plataforma distribuida (Spark/Hadoop) para su volumen.

**Dataset:** Emotion Detection from Text (Kaggle, pashupatigupta): https://www.kaggle.com/datasets/pashupatigupta/emotion-detection-from-text (tweet_emotions.csv: 40 000 tweets, columnas tweet_id, sentiment, content)

## Desafío

1) Recomendar la **granularidad de emociones** (número de clases) que el área puede usar con confianza, con evidencia del experimento de clases y una línea base. 2) Recomendar el **modelo** (Naive Bayes, SVM o RNN) con su costo y su desempeño. 3) Estimar si su volumen justifica **Spark** y cómo se desplegaría (HDFS/S3, clúster en AWS), con evidencia del pipeline en PySpark. 4) Declarar una limitación seria del dataset y una implicación ética.

> **Formato de entrega:** un único `.ipynb` ejecutado. Código limpio y comentado. Antes de cada gráfico,
> una línea que diga qué se analiza; después, la conclusión de negocio. Nombre del archivo:
> `grupoN_Semana3_Ejercicio.ipynb`.

In [ ]:
# ===== Entorno: el mismo notebook corre en Google Colab y en local =====
import importlib, importlib.util, os, shutil, subprocess, sys, urllib.request   # utilidades: instalar paquetes, rutas y descargas
from pathlib import Path                            # rutas iguales en Windows, Linux y Colab

EN_COLAB = "google.colab" in sys.modules           # True si estamos dentro de Colab
print("Entorno:", "Google Colab" if EN_COLAB else "local", "| Python", sys.version.split()[0])   # dónde corre y qué Python

# Instalamos solo los paquetes que falten. El nombre en pip no siempre es el del import:
# ej.: pip install scikit-learn -> import sklearn; nx-parallel -> nx_parallel
IMPORT_DE = {"scikit-learn": "sklearn", "sentence-transformers": "sentence_transformers",   # nombre en pip -> nombre del import
             "delta-spark": "delta", "python-calamine": "python_calamine",   # instaladores y lectores de Excel
             "python-louvain": "community", "imbalanced-learn": "imblearn"}   # comunidades y desbalance de clases
for pip_nombre in ['pyspark', 'spacy', 'kagglehub', 'bertopic', 'sentence-transformers']:   # paquetes que necesita esta práctica
    base = pip_nombre.split("==")[0].split(">=")[0]                    # ej.: "networkx>=3.7" -> "networkx"
    modulo = IMPORT_DE.get(base, base.replace("-", "_"))   # nombre del import. Ej.: nx-parallel -> nx_parallel
    if importlib.util.find_spec(modulo) is None:   # solo si NO está instalado (Colab ya trae muchos)
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_nombre])   # Colab y venv con pip
        if r.returncode != 0 and shutil.which("uv"):                   # venv creado con uv (no trae pip)
            r = subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, pip_nombre])   # instala con uv en este mismo Python
        if r.returncode != 0:   # ni pip ni uv pudieron instalarlo
            raise RuntimeError(f"Instala '{pip_nombre}' en este entorno (pip/uv) y vuelve a ejecutar")   # error explícito con la acción a seguir
        importlib.invalidate_caches()                                  # para que el import vea el paquete recién instalado

# Spark corre sobre la JVM: en Colab instalamos Java 17; en local debe existir JAVA_HOME
if EN_COLAB and not os.environ.get("JAVA_HOME"):   # solo en Colab y si Java aún no está
    subprocess.run("apt-get install -y -qq openjdk-17-jdk-headless > /dev/null", shell=True, check=True)   # instala Java 17 sin mensajes (tarda ~1 min)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"   # dónde lo deja apt en Colab
if not EN_COLAB and not os.environ.get("JAVA_HOME") and shutil.which("java") is None:   # local sin JAVA_HOME ni java en el PATH
    # Buscamos un JDK 17 en rutas típicas. Ej.: ~/opt/jdk17 (WSL), /usr/lib/jvm/java-17-openjdk-amd64 (Ubuntu)
    import glob   # glob busca rutas con comodines. Ej.: /usr/lib/jvm/*17*
    for jdk in [str(Path.home() / "opt" / "jdk17"), *glob.glob("/usr/lib/jvm/*17*"),   # rutas candidatas de un JDK 17
                *glob.glob(r"C:\Program Files\Eclipse Adoptium\jdk-17*")]:   # Windows con Temurin 17
        if Path(jdk, "bin").exists():   # es un JDK si tiene la carpeta bin
            os.environ["JAVA_HOME"] = jdk                                    # Spark lo lee al arrancar
            os.environ["PATH"] = str(Path(jdk, "bin")) + os.pathsep + os.environ["PATH"]   # el comando java queda disponible
            break   # se usa el primero que exista
if not EN_COLAB and not os.environ.get("JAVA_HOME") and shutil.which("java") is None:   # no se encontró ningún JDK
    raise RuntimeError("Spark necesita Java 17: instálalo (por ejemplo, Eclipse Temurin 17), define JAVA_HOME "   # sin Java, Spark no arranca
                       "y reinicia el kernel")   # acción para corregirlo
print("JAVA_HOME:", os.environ.get("JAVA_HOME", "(java en el PATH)"))   # confirma qué Java usará Spark
# Los workers de Spark deben usar el MISMO Python que este notebook (si no: PYTHON_VERSION_MISMATCH)
os.environ["PYSPARK_PYTHON"] = os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable   # mismo Python en driver y workers
# Carpeta de datos relativa: en Colab es /content/data, en local ./data
DATA = Path("data"); DATA.mkdir(exist_ok=True)   # crea la carpeta si no existe

def descargar(url, nombre):   # Ej.: descargar(URL, "telco.csv") -> data/telco.csv
    # Descarga una sola vez; si el archivo ya existe, no vuelve a bajarlo
    destino = DATA / nombre   # ruta local del archivo. Ej.: data/telco.csv
    if not destino.exists():   # descarga solo la primera vez
        print("Descargando", nombre, "...")   # aviso de progreso
        urllib.request.urlretrieve(url, destino)   # descarga el archivo en data/
    return destino   # devuelve la ruta para usarla después

ARCHIVOS = [   # lista de (url, nombre) que usa esta práctica
    
]
for url, nombre in ARCHIVOS:   # descarga cada archivo
    print(nombre, "->", round(descargar(url, nombre).stat().st_size / 1e6, 2), "MB")   # y muestra su tamaño en MB

In [ ]:
# ===== Estilo de gráficos (principios de Tufte): alta proporción dato-tinta, un solo color de acento =====
import matplotlib.pyplot as plt                     # gráficos
from cycler import cycler                           # orden de colores por defecto
TINTA, ACENTO, GRIS = "#14323F", "#6E5BD6", "#B7C3C9"     # tokens del módulo; ACENTO = color de esta semana
plt.rcParams.update({   # estilo por defecto de todos los gráficos
    "axes.spines.top": False, "axes.spines.right": False,    # sin marco: solo los ejes necesarios
    "axes.grid": False, "axes.edgecolor": GRIS,              # sin rejilla (las etiquetas dicen el valor)
    "axes.titlelocation": "left", "axes.titleweight": "bold", "axes.titlecolor": TINTA,   # títulos a la izquierda y en negrita
    "axes.prop_cycle": cycler(color=[TINTA, ACENTO, GRIS, "#5B6B73"]),
    "font.size": 12, "legend.frameon": False, "figure.dpi": 110,   # letra legible, leyenda sin caja, nitidez
})
# Regla práctica: pinten con ACENTO solo el dato que quieren que el lector mire; el resto en GRIS.

In [ ]:
# ===== Reproducibilidad: semilla y versiones de las librerías =====
import os, sys, platform
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")   # menos mensajes de TensorFlow
import numpy, pandas, sklearn, spacy, pyspark
SEMILLA = 42   # semilla única de todo el notebook
print("Semilla:", SEMILLA)
print("Python", sys.version.split()[0], "|", platform.platform())
for lib in (numpy, pandas, sklearn, spacy, pyspark):
    print(f"{lib.__name__:<10}", lib.__version__)
import tensorflow as tf
print("tensorflow", tf.__version__)

<a id="fase-1"></a>
## Fase 1: Carga, limpieza y exploración del corpus

Paso 1 del enunciado. Carguen el CSV, revisen duplicados, longitud de los textos y la distribución de las 13 emociones (¿qué tan desbalanceada está? ¿qué significa la etiqueta `empty`?). Limpien menciones, URLs y entidades HTML, y **documenten** cada decisión de limpieza.

> **Pregunta de la guía:** Duplicados, longitud media y distribución de las 13 emociones. ¿Qué significa «empty» y qué hicieron con ella?

**Qué analizamos:** Se carga el corpus de 40 000 tweets con 13 emociones, se explora ANTES de limpiar (duplicados, longitud, desbalance de clases y ruido de Twitter: menciones, URLs y entidades HTML) y luego se limpia documentando cada decisión con su impacto medido. También se examina qué significa la etiqueta `empty`.

In [ ]:
import re, html, time, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import spacy
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
SEMILLA = 42
np.random.seed(SEMILLA)
Path("results").mkdir(exist_ok=True); Path("figuras").mkdir(exist_ok=True)
t_ini_fase1 = time.time()

# --- Carga: Kaggle (kagglehub) si hay acceso; si no, copia local data/tweet_emotions.csv ---
def cargar_csv():
    try:
        import kagglehub
        ruta = kagglehub.dataset_download('pashupatigupta/emotion-detection-from-text')
        return pd.read_csv(f'{ruta}/tweet_emotions.csv')
    except Exception:
        return pd.read_csv('data/tweet_emotions.csv')
df_crudo = cargar_csv()
print("Forma del corpus crudo:", df_crudo.shape)
df_crudo.head()

**Qué se analiza:** duplicados del corpus crudo (por `tweet_id`, por texto y por texto con etiquetas contradictorias). Un texto repetido con emociones distintas es ruido de etiquetado y puede contaminar el entrenamiento y la evaluación.

In [ ]:
# --- Exploración sobre el corpus CRUDO (antes de limpiar) ---
n_total = len(df_crudo)
dup_id = int(df_crudo['tweet_id'].duplicated().sum())             # mismo id repetido
dup_texto = int(df_crudo['content'].duplicated().sum())           # filas repetidas en texto (sin contar la 1.ª copia)
# textos que aparecen con más de una emoción distinta
conflicto_crudo = df_crudo.groupby('content')['sentiment'].transform('nunique') > 1
filas_conflicto = int(conflicto_crudo.sum())
textos_conflicto = int(df_crudo.loc[conflicto_crudo, 'content'].nunique())
print(f"Filas: {n_total}")
print(f"Duplicados por tweet_id: {dup_id}")
print(f"Duplicados por texto (excluyendo 1.ª copia): {dup_texto}")
print(f"Textos con etiquetas contradictorias: {textos_conflicto} textos ({filas_conflicto} filas)")
print("Nulos por columna:\n", df_crudo.isna().sum().to_string())
df_crudo.loc[conflicto_crudo].sort_values('content').head(6)[['content', 'sentiment']]

**Qué se analiza:** longitud de los tweets (caracteres y palabras) con estadísticos y su histograma, para conocer el tamaño típico del texto que verán los modelos.

In [ ]:
# Longitud en caracteres y en palabras (sobre el texto crudo)
largo_car = df_crudo['content'].str.len()
largo_pal = df_crudo['content'].str.split().str.len()
resumen_largo = pd.DataFrame({
    'caracteres': [largo_car.mean(), largo_car.median(), largo_car.min(), largo_car.max()],
    'palabras':   [largo_pal.mean(), largo_pal.median(), largo_pal.min(), largo_pal.max()],
}, index=['media', 'mediana', 'min', 'max']).round(1)
print(resumen_largo)

TINTA, ACENTO, GRIS = "#14323F", "#6E5BD6", "#B7C3C9"
def estilo(ax):
    # sin rejilla y sin marco superior/derecho
    ax.grid(False)
    ax.spines[['top', 'right']].set_visible(False)

fig, ejes = plt.subplots(1, 2, figsize=(11, 3.8))
ejes[0].hist(largo_car, bins=40, color=GRIS)
ejes[0].axvline(largo_car.mean(), color=ACENTO, lw=2, label=f"media = {largo_car.mean():.0f}")
ejes[0].set_xlabel("Caracteres por tweet"); ejes[0].set_ylabel("Tweets"); ejes[0].legend(frameon=False)
ejes[1].hist(largo_pal, bins=range(0, 40), color=GRIS)
ejes[1].axvline(largo_pal.mean(), color=ACENTO, lw=2, label=f"media = {largo_pal.mean():.1f}")
ejes[1].set_xlabel("Palabras por tweet"); ejes[1].legend(frameon=False)
for ax in ejes: estilo(ax)
plt.tight_layout()
plt.savefig("figuras/fase1_longitud.png", dpi=150)
plt.show()

**Qué se analiza:** distribución de las 13 emociones (barras horizontales ordenadas) y la razón entre la clase más grande y la más pequeña, que mide el desbalance.

In [ ]:
conteo = df_crudo['sentiment'].value_counts()
razon_desbalance = conteo.max() / conteo.min()
print(conteo.to_frame('tweets').assign(porcentaje=(conteo / n_total * 100).round(1)))
print(f"\nRazón mayor/menor clase: {conteo.idxmax()} ({conteo.max()}) / {conteo.idxmin()} ({conteo.min()}) = {razon_desbalance:.1f}x")

orden = conteo.sort_values()   # de menor a mayor para que la mayor quede arriba
colores = [ACENTO if e == 'empty' else GRIS for e in orden.index]   # ACENTO solo para el dato clave: empty
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(orden.index, orden.values, color=colores)
for i, v in enumerate(orden.values):
    ax.text(v + 80, i, f"{v:,}", va='center', color=TINTA, fontsize=9)
ax.set_xlabel("Número de tweets")
estilo(ax)
plt.tight_layout()
plt.savefig("figuras/fase1_distribucion_emociones.png", dpi=150)
plt.show()

**Qué se analiza:** qué significa la etiqueta `empty`. Se comparan sus ejemplos y su longitud contra el resto del corpus para decidir si es un dato faltante o una categoría con contenido.

In [ ]:
vacia = df_crudo[df_crudo['sentiment'] == 'empty']
print("Ejemplos de tweets con etiqueta 'empty':")
for t in vacia['content'].sample(12, random_state=SEMILLA):
    print("  -", t)
# ¿tiene el texto vacío o nulo? (si fuera dato faltante, el texto estaría vacío)
print("\n'empty' con texto nulo o vacío:", int((vacia['content'].isna() | (vacia['content'].str.strip() == '')).sum()))
print("Longitud media 'empty':", round(vacia['content'].str.len().mean(), 1),
      "| resto:", round(df_crudo.loc[df_crudo['sentiment'] != 'empty', 'content'].str.len().mean(), 1))
# ¿es empty simplemente texto de otra clase? Comparación con neutral
print("Menciones (@) en 'empty':", round(vacia['content'].str.contains('@').mean() * 100, 1), "% | en neutral:",
      round(df_crudo.loc[df_crudo['sentiment'] == 'neutral', 'content'].str.contains('@').mean() * 100, 1), "%")

**Qué se analiza:** cuánto ruido propio de Twitter hay en el texto crudo (menciones, URLs, entidades HTML, hashtags) para justificar las decisiones de limpieza.

In [ ]:
# Conteos de ruido sobre el corpus crudo (tweets que contienen cada elemento)
patron_mencion = r'@\w+'
patron_url = r'https?://\S+|www\.\S+'
patron_html = r'&(?:#\d+|\w+);'
n_mencion = int(df_crudo['content'].str.contains(patron_mencion).sum())
n_url = int(df_crudo['content'].str.contains(patron_url).sum())
n_html = int(df_crudo['content'].str.contains(patron_html).sum())
n_hashtag = int(df_crudo['content'].str.contains('#').sum())
print("Tweets con menciones:", n_mencion, f"({n_mencion / n_total * 100:.1f}%)")
print("Tweets con URLs:", n_url, f"({n_url / n_total * 100:.1f}%)")
print("Tweets con entidades HTML:", n_html, f"({n_html / n_total * 100:.1f}%)")
print("Tweets con '#':", n_hashtag, f"({n_hashtag / n_total * 100:.1f}%)")
print("Entidades HTML más frecuentes:\n", df_crudo['content'].str.findall(patron_html).explode().value_counts().head(5).to_string())

In [ ]:
# --- Limpieza (decisiones documentadas en la tabla de más abajo) ---
def limpiar(t):
    t = html.unescape(t)                       # &amp; -> &, &quot; -> "
    t = re.sub(r'https?://\S+|www\.\S+', ' ', t)   # URLs
    t = re.sub(r'@\w+', ' ', t)                # menciones @usuario
    t = t.replace('#', '')                     # hashtag: conserva la palabra
    return re.sub(r'\s+', ' ', t).strip()      # espacios múltiples

df = df_crudo.copy()
# duplicados: textos con etiquetas contradictorias se eliminan todos; el resto, se conserva la 1.ª copia
conflicto = df.groupby('content')['sentiment'].transform('nunique') > 1
df = df[~conflicto].drop_duplicates('content').copy()
df['texto_limpio'] = df['content'].map(limpiar)
df = df[df['texto_limpio'].str.len() > 0].reset_index(drop=True)   # tweets que quedaron vacíos (solo menciones/URL)
ETIQUETAS = sorted(df['sentiment'].unique())

**Qué se analiza:** impacto de cada decisión de limpieza (cuántos tweets afecta) y el tamaño final del corpus.

In [ ]:
# Medición del impacto de cada decisión
sin_conflicto = df_crudo[~conflicto]
dup_exactos_restantes = int(sin_conflicto['content'].duplicated().sum())
n_cambia_mencion = n_mencion
quedan_vacios = n_total - filas_conflicto - dup_exactos_restantes - len(df)  # vacíos entre los que sobrevivían a duplicados
tabla_decisiones = pd.DataFrame([
    ["Eliminar textos con etiquetas contradictorias", "El mismo texto con 2+ emociones no tiene respuesta correcta única; se descartan todas sus copias", filas_conflicto],
    ["Conservar solo la 1.ª copia de textos duplicados", "Copias idénticas inflan métricas y pueden quedar en entrenamiento y prueba a la vez (fuga de datos)", dup_exactos_restantes],
    ["Decodificar entidades HTML (&quot;, &amp;, &lt;, &gt;)", "Son artefactos de la exportación; el modelo debe ver '\"' y '&', no códigos", n_html],
    ["Quitar URLs", "Direcciones únicas sin valor emocional; solo añaden vocabulario raro", n_url],
    ["Quitar menciones @usuario", "Nombres de usuario no expresan emoción y fragmentan el vocabulario", n_mencion],
    ["Quitar '#' y conservar la palabra", "El hashtag suele llevar el tema o emoción (p. ej. #fail)", n_hashtag],
    ["Colapsar espacios múltiples", "Normaliza el texto tras las eliminaciones", int(df_crudo['content'].str.contains(r'\s{2,}').sum())],
    ["Descartar tweets que quedan vacíos tras limpiar", "Si solo había menciones/URLs no queda texto que clasificar", quedan_vacios],
    ["NO pasar a minúsculas ni quitar emoticonos/puntuación aquí", "Se deja para cada modelo (vectorizadores y spaCy lo gestionan; emoticonos como :( aportan señal)", 0],
    ["Conservar la clase `empty` como 13.ª emoción", "La guía pide matriz de 13 clases; es una etiqueta ruidosa, no dato faltante", int((df_crudo['sentiment'] == 'empty').sum())],
], columns=['decisión', 'por qué', 'tweets afectados'])
pd.set_option('display.max_colwidth', 110)
print(tabla_decisiones.to_string(index=False))
n_final = len(df)
perdidas = n_total - n_final
print(f"\nCorpus: {n_total} -> {n_final} tweets tras limpiar ({perdidas} eliminados, {perdidas / n_total * 100:.2f}%)")
print("Clases tras limpiar:", len(ETIQUETAS), "| 'empty' conservada:", int((df['sentiment'] == 'empty').sum()))
print("Longitud media tras limpiar: %.1f caracteres, %.1f palabras" % (df['texto_limpio'].str.len().mean(), df['texto_limpio'].str.split().str.len().mean()))
cambia = df['content'] != df['texto_limpio']
print("Tweets modificados por la limpieza:", int(cambia.sum()), f"({cambia.mean() * 100:.1f}%)")
print(df.loc[cambia, ['content', 'texto_limpio']].head(4).to_string(index=False))

In [ ]:
# Guardar cifras clave de la fase
conteo_limpio = df['sentiment'].value_counts()
resultados_fase1 = {
    "filas_crudas": int(n_total), "filas_limpias": int(n_final),
    "dup_tweet_id": dup_id, "dup_texto": dup_texto,
    "textos_contradictorios": textos_conflicto, "filas_contradictorias": filas_conflicto,
    "dup_exactos_tras_conflicto": dup_exactos_restantes, "vacios_tras_limpiar": int(quedan_vacios),
    "longitud_caracteres": {k: float(v) for k, v in zip(['media', 'mediana', 'min', 'max'], resumen_largo['caracteres'])},
    "longitud_palabras": {k: float(v) for k, v in zip(['media', 'mediana', 'min', 'max'], resumen_largo['palabras'])},
    "distribucion_cruda": {k: int(v) for k, v in conteo.items()},
    "distribucion_limpia": {k: int(v) for k, v in conteo_limpio.items()},
    "razon_desbalance_cruda": float(razon_desbalance),
    "razon_desbalance_limpia": float(conteo_limpio.max() / conteo_limpio.min()),
    "pct_neutral_worry": float((conteo['neutral'] + conteo['worry']) / n_total * 100),
    "tweets_con_menciones": n_mencion, "tweets_con_urls": n_url, "tweets_con_html": n_html, "tweets_con_hashtag": n_hashtag,
    "empty_n": int(len(vacia)), "empty_pct": float(len(vacia) / n_total * 100),
    "tabla_decisiones": tabla_decisiones.to_dict(orient='records'),
    "tiempo_s": round(time.time() - t_ini_fase1, 1),
}
with open("results/fase1.json", "w") as f:
    json.dump(resultados_fase1, f, ensure_ascii=False, indent=2)
print("Guardado results/fase1.json")

**Conclusión de negocio (Fase 1).**
- El corpus tiene 40 000 tweets sin nulos ni `tweet_id` repetidos, pero 173 textos duplicados y 62 textos (207 filas) con emociones contradictorias: se descartan esas 207 filas y las copias repetidas (28), y 77 tweets más quedan vacíos tras limpiar. Quedan 39 688 tweets (0,78 % menos).
- El tweet típico es corto: media de 73,4 caracteres y 13,2 palabras (mediana 69 y 12; máximo 167 caracteres), por lo que los modelos trabajan con muy poco contexto por ejemplo.
- El desbalance es fuerte: `neutral` (8 638) y `worry` (8 459) suman 42,7 % del corpus, mientras `anger` (110) y `boredom` (179) apenas llegan a 0,3 % y 0,4 %. La razón mayor/menor es 78,5 veces: un modelo que solo prediga `neutral` ya acierta 21,6 % en el corpus crudo (21,5 % tras limpiar), así que la métrica correcta es F1 macro y no la exactitud.
- `empty` (827 tweets, 2,1 %) NO es dato faltante: ningún texto está vacío; son tweets con texto normal a los que el anotador no asignó una emoción identificable (algunos incluso expresan aburrimiento o enfado). Es una etiqueta ruidosa; se conserva como 13.ª clase y se espera que sea de las más difíciles de predecir.
- El texto crudo es ruidoso: 47,4 % de los tweets tiene menciones, 5,8 % entidades HTML y 4,8 % URLs. La limpieza modifica el 66,3 % de los tweets y reduce la longitud media a 65,6 caracteres (12,7 palabras), sin perder ninguna de las 13 clases.

<a id="fase-2"></a>
## Fase 2: Análisis lingüístico con spaCy: POS y NER

Sobre una muestra estratificada: tokenización, lematización, **etiquetado gramatical (POS)**, **reconocimiento de entidades (NER)** y **análisis de dependencias** (al menos 3 tweets, con `displacy`). Comparen la proporción de adjetivos, verbos e interjecciones entre emociones y las entidades más frecuentes. ¿Aporta información útil para clasificar?

> **Pregunta de la guía:** ¿Qué categorías gramaticales, entidades y dependencias distinguen a las emociones? ¿Aportan para clasificar?

**Qué analizamos:** Se analiza la estructura lingüística de una muestra estratificada de tweets con spaCy (`en_core_web_sm`): tokenización, lematización, etiquetado gramatical (POS), reconocimiento de entidades (NER) y árbol de dependencias. Se compara la proporción de adjetivos, verbos e interjecciones entre emociones y se mide con un clasificador simple si estas características ayudan a clasificar.

In [ ]:
import time, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import spacy
from collections import Counter
from scipy.stats import kruskal
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
t_ini_fase2 = time.time()

# Muestra estratificada: hasta 200 tweets por emoción (anger y boredom tienen menos de 200: se toman todos)
partes = [g.sample(min(200, len(g)), random_state=SEMILLA) for _, g in df.groupby('sentiment')]
muestra = pd.concat(partes).reset_index(drop=True)
print("Muestra:", muestra.shape)
print(muestra['sentiment'].value_counts().to_string())

# Pipeline sin el parser (más rápido): tokenizador + tagger + lematizador + NER
nlp = spacy.load('en_core_web_sm', disable=['parser'])
t0 = time.time()
docs = list(nlp.pipe(muestra['texto_limpio'], batch_size=500))
print(f"spaCy procesó {len(docs)} tweets en {time.time() - t0:.1f} s")

**Qué se analiza:** cómo spaCy tokeniza y lematiza el texto informal de los tweets y qué etiqueta gramatical (POS) asigna a cada palabra.

In [ ]:
# Ejemplos de tokenización, lematización y POS (3 tweets de la muestra)
for i in [0, 700, 1500]:
    doc = docs[i]
    print(f"[{muestra.loc[i, 'sentiment']}] {doc.text}")
    print(pd.DataFrame({'token': [t.text for t in doc], 'lema': [t.lemma_ for t in doc],
                        'POS': [t.pos_ for t in doc]}).T.to_string(header=False))
    print()
# Tokens cuyo lema difiere del texto (la lematización agrupa formas de una misma palabra)
cambios = Counter((t.text.lower(), t.lemma_.lower()) for d in docs for t in d
                  if t.is_alpha and t.text.lower() != t.lemma_.lower())
print("Pares palabra -> lema más frecuentes:", cambios.most_common(12))

**Qué se analiza:** proporción de adjetivos (ADJ), verbos (VERB) e interjecciones (INTJ) sobre el total de tokens de cada tweet, promediada por emoción. Si una emoción usa más un tipo de palabra, esa diferencia podría servir al clasificador.

In [ ]:
# Proporción de cada categoría gramatical por tweet (sobre tokens que no son espacios)
categorias = ['ADJ', 'VERB', 'INTJ', 'NOUN', 'ADV', 'PRON', 'PROPN', 'ADP', 'AUX', 'DET', 'NUM', 'PUNCT', 'SYM', 'X']
filas = []
for doc in docs:
    toks = [t.pos_ for t in doc if not t.is_space]
    n = max(len(toks), 1)
    conteo_pos = Counter(toks)
    filas.append({c: conteo_pos.get(c, 0) / n for c in categorias})
prop_pos = pd.DataFrame(filas)
prop_pos['sentiment'] = muestra['sentiment'].values
por_emocion = prop_pos.groupby('sentiment')[['ADJ', 'VERB', 'INTJ']].mean() * 100   # en %
por_emocion = por_emocion.round(2)
print(por_emocion.sort_values('INTJ', ascending=False).to_string())

# Magnitud de las diferencias: rango entre emociones y prueba de Kruskal-Wallis (¿difiere alguna emoción?)
tabla_dif = []
for c in ['ADJ', 'VERB', 'INTJ']:
    grupos = [g[c].values for _, g in prop_pos.groupby('sentiment')]
    H, p = kruskal(*grupos)
    eta2 = max((H - len(grupos) + 1) / (len(prop_pos) - len(grupos)), 0)   # tamaño de efecto (eta² de Kruskal)
    tabla_dif.append({'categoria': c, 'media_global_%': prop_pos[c].mean() * 100,
                      'max_emocion': por_emocion[c].idxmax(), 'max_%': por_emocion[c].max(),
                      'min_emocion': por_emocion[c].idxmin(), 'min_%': por_emocion[c].min(),
                      'dif_puntos_pp': por_emocion[c].max() - por_emocion[c].min(), 'p_kruskal': p, 'eta2': eta2})
tabla_dif = pd.DataFrame(tabla_dif).round(4)
print(tabla_dif.to_string(index=False))

TINTA, ACENTO, GRIS = "#14323F", "#6E5BD6", "#B7C3C9"
fig, ejes = plt.subplots(1, 3, figsize=(13, 5), sharey=False)
for ax, c in zip(ejes, ['ADJ', 'VERB', 'INTJ']):
    serie = por_emocion[c].sort_values()
    colores = [ACENTO if e == serie.idxmax() else GRIS for e in serie.index]   # ACENTO: la emoción con mayor proporción
    ax.barh(serie.index, serie.values, color=colores)
    ax.axvline(prop_pos[c].mean() * 100, color=TINTA, lw=1, ls='--')   # media global
    ax.set_title(f"{c} (% de tokens)", color=TINTA)
    ax.grid(False); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig("figuras/fase2_pos_por_emocion.png", dpi=150)
plt.show()

**Qué se analiza:** qué entidades nombradas (personas, organizaciones, lugares, fechas) aparecen más en la muestra y qué tan frecuentes son por emoción.

In [ ]:
# Entidades nombradas (NER)
entidades = [(e.text, e.label_, s) for doc, s in zip(docs, muestra['sentiment']) for e in doc.ents]
tabla_ent = pd.DataFrame(entidades, columns=['entidad', 'tipo', 'emocion'])
print("Entidades totales:", len(tabla_ent), "| tweets con al menos una:", sum(len(d.ents) > 0 for d in docs), "de", len(docs))
top10 = tabla_ent['entidad'].str.lower().value_counts().head(10)
print("\nTop 10 entidades:\n", top10.to_string())
print("\nTipos de entidad:\n", tabla_ent['tipo'].value_counts().head(8).to_string())
for tipo in ['PERSON', 'ORG', 'GPE', 'DATE']:
    top_tipo = tabla_ent.loc[tabla_ent['tipo'] == tipo, 'entidad'].str.lower().value_counts().head(5)
    print(f"\nTop {tipo}:", top_tipo.to_dict())
# Densidad de entidades por emoción: % de tweets con al menos una entidad
con_ent = pd.Series([len(d.ents) > 0 for d in docs]).groupby(muestra['sentiment'].values).mean() * 100
print("\n% de tweets con entidad por emoción:\n", con_ent.sort_values(ascending=False).round(1).to_string())

**Qué se analiza:** el árbol de dependencias de 3 tweets (qué palabra depende de cuál). Para esto se usa el modelo completo de spaCy, con parser.

In [ ]:
# Dependencias: modelo completo (con parser) solo en 3 tweets cortos de emociones distintas
nlp_dep = spacy.load('en_core_web_sm')
from spacy import displacy
elegidos = []
for emo in ['happiness', 'hate', 'worry']:
    cand = muestra[(muestra['sentiment'] == emo) & (muestra['texto_limpio'].str.split().str.len().between(6, 11))]
    elegidos.append(cand.index[0])
for k, i in enumerate(elegidos):
    doc = nlp_dep(muestra.loc[i, 'texto_limpio'])
    print(f"\n[{muestra.loc[i, 'sentiment']}] {doc.text}")
    print(pd.DataFrame({'token': [t.text for t in doc], 'dep': [t.dep_ for t in doc],
                        'cabeza': [t.head.text for t in doc]}).T.to_string(header=False))
    opciones = {'compact': True, 'distance': 90}
    svg = displacy.render(doc, style='dep', jupyter=False, options=opciones)
    with open(f"figuras/fase2_dependencias_{k + 1}.svg", "w", encoding="utf-8") as f:
        f.write(svg)
    displacy.render(doc, style='dep', jupyter=True, options=opciones)   # se dibuja en el notebook

**Qué se analiza:** las etiquetas de dependencia sobre toda la muestra (parser completo): negaciones (`neg`), sujetos, interjecciones y modificadores. Se comparan entre emociones igual que el POS.

In [ ]:
# Parser completo sobre toda la muestra (solo para contar relaciones de dependencia)
t0 = time.time()
docs_dep = list(nlp_dep.pipe(muestra['texto_limpio'], batch_size=500))
print(f"Parser completo: {time.time() - t0:.1f} s")
deps_interes = ['neg', 'nsubj', 'dobj', 'amod', 'advmod', 'intj', 'aux', 'prep', 'ROOT']
filas = []
for doc in docs_dep:
    n = max(sum(1 for t in doc if not t.is_space), 1)
    c = Counter(t.dep_ for t in doc)
    filas.append({d: c.get(d, 0) / n for d in deps_interes})
prop_dep = pd.DataFrame(filas)
prop_dep['sentiment'] = muestra['sentiment'].values
dep_emocion = (prop_dep.groupby('sentiment')[['neg', 'intj', 'amod', 'advmod']].mean() * 100).round(2)
print(dep_emocion.sort_values('neg', ascending=False).to_string())
# ¿Difiere la negación entre emociones? (misma prueba y tamaño de efecto que con POS)
grupos_neg = [g['neg'].values for _, g in prop_dep.groupby('sentiment')]
H_neg, p_neg = kruskal(*grupos_neg)
eta2_neg = max((H_neg - len(grupos_neg) + 1) / (len(prop_dep) - len(grupos_neg)), 0)
print(f"Negación (neg): Kruskal-Wallis p = {p_neg:.2g}, eta2 = {eta2_neg:.3f}")

**Qué se analiza:** si estas características gramaticales sirven para clasificar. Se entrena una regresión logística SOLO con proporciones POS (y con POS + dependencias + entidades) con validación cruzada de 5 particiones y se compara con el azar y con la clase mayoritaria (macro-F1). Como referencia se añade una bolsa de palabras TF-IDF en la misma muestra.

In [ ]:
y_m = muestra['sentiment'].to_numpy(dtype=str)
X_pos = prop_pos[categorias].values
X_pos_dep = np.hstack([X_pos, prop_dep[deps_interes].values,
                       np.array([[len(d.ents) > 0, len(d.ents)] for d in docs])])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

def f1_cv(modelo, X):
    # macro-F1 medio en 5 particiones estratificadas
    return cross_val_score(modelo, X, y_m, cv=cv, scoring='f1_macro')

lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEMILLA))
# Azar: macro-F1 de predicciones aleatorias uniformes sobre las 13 clases (50 repeticiones)
from sklearn.metrics import f1_score
rng = np.random.default_rng(SEMILLA)
clases_m = np.unique(y_m)
f1_azar = np.array([f1_score(y_m, rng.choice(clases_m, size=len(y_m)), average='macro') for _ in range(50)])
res_clf = {
    'mayoritaria': f1_cv(DummyClassifier(strategy='most_frequent'), X_pos),
    'azar (uniforme, 50 repeticiones)': f1_azar,
    'LogReg solo proporciones POS': f1_cv(lr(), X_pos),
    'LogReg POS + dependencias + entidades': f1_cv(lr(), X_pos_dep),
    'LogReg bolsa de palabras TF-IDF (referencia)': f1_cv(make_pipeline(TfidfVectorizer(min_df=2, ngram_range=(1, 2)),
                                      LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEMILLA)), muestra['texto_limpio']),
}
tabla_clf = pd.DataFrame({k: {'f1_macro': v.mean(), 'desv': v.std()} for k, v in res_clf.items()}).T.round(3)
print(tabla_clf.to_string())

fig, ax = plt.subplots(figsize=(8, 3.6))
nombres = list(tabla_clf.index)[::-1]
vals = tabla_clf.loc[nombres, 'f1_macro']
ax.barh(nombres, vals, xerr=tabla_clf.loc[nombres, 'desv'], color=[ACENTO if n == 'LogReg solo proporciones POS' else GRIS for n in nombres])
ax.set_xlabel("F1 macro (validación cruzada, 5 particiones)")
ax.grid(False); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig("figuras/fase2_clasificacion_pos.png", dpi=150)
plt.show()

In [ ]:
# Guardar cifras clave de la fase
resultados_fase2 = {
    "tamano_muestra": int(len(muestra)),
    "por_emocion_ADJ_VERB_INTJ_pct": por_emocion.to_dict(orient='index'),
    "diferencias": tabla_dif.to_dict(orient='records'),
    "entidades_totales": int(len(tabla_ent)),
    "tweets_con_entidad_pct": float(np.mean([len(d.ents) > 0 for d in docs]) * 100),
    "top10_entidades": top10.to_dict(),
    "tipos_entidad": tabla_ent['tipo'].value_counts().head(8).to_dict(),
    "dep_por_emocion_pct": dep_emocion.to_dict(orient='index'),
    "neg_kruskal_p": float(p_neg), "neg_eta2": float(eta2_neg),
    "f1_macro_cv": {k: {"media": float(v.mean()), "desv": float(v.std())} for k, v in res_clf.items()},
    "tiempo_s": round(time.time() - t_ini_fase2, 1),
}
with open("results/fase2.json", "w") as f:
    json.dump(resultados_fase2, f, ensure_ascii=False, indent=2, default=float)
print("Guardado results/fase2.json")

**Conclusión de negocio (Fase 2).** Muestra estratificada de 2 489 tweets (200 por emoción; `boredom` 179 y `anger` 110, todos los disponibles).
- Las diferencias gramaticales existen pero son pequeñas. Adjetivos: de 4,59 % (`neutral`) a 7,16 % (`surprise`), media 6,13 %. Verbos: de 11,37 % (`love`) a 14,40 % (`hate`), media 12,81 %. Ambas son estadísticamente detectables (Kruskal-Wallis p = 0,005 y 0,003) pero con tamaño de efecto minúsculo (eta² cercano a 0,007, es decir, la emoción explica menos del 1 % de la variación entre tweets). Las interjecciones (media 1,55 %; `empty` 2,80 % vs `anger` 0,83 %) NO difieren de forma significativa (p = 0,66).
- Las dependencias dan la señal gramatical más nítida (negación: Kruskal-Wallis p = 1,8e-11, eta² = 0,026, aun así pequeño): la negación (`neg`) pesa 2,39 % de los tokens en `worry` y 2,25 % en `hate` frente a 0,62 % en `happiness` y 0,75 % en `love`; los modificadores adjetivales (`amod`) son más frecuentes en `happiness` (4,34 %) y `love` (4,22 %) que en `neutral` (2,85 %). Es decir, las emociones positivas describen con adjetivos y las negativas niegan o se preocupan por acciones.
- NER aporta poco en tweets informales: 1 926 entidades en 1 228 de 2 489 tweets (49,3 %), dominadas por DATE (447: today, tonight, tomorrow), PERSON (423) y ORG (333), con errores evidentes ('awww' como PERSON, 'lol' como ORG). Las entidades más frecuentes (today 98, tonight 48, tomorrow 33) son marcas temporales, no emociones. Los tweets con entidad van de 60,0 % (`happiness`) a 39,0 % (`worry`).
- ¿Aporta para clasificar? Muy poco. Con validación cruzada (5 particiones), regresión logística con solo proporciones POS logra F1 macro 0,095 (+- 0,011), frente a 0,076 del azar uniforme y 0,011 de la clase mayoritaria; añadir dependencias y entidades sube solo a 0,098. Una bolsa de palabras TF-IDF en la misma muestra llega a 0,184, el doble. Decisión: las características gramaticales se pueden usar como complemento (la negación es la más informativa), pero el contenido léxico es la señal principal para clasificar emociones.

<a id="fase-3"></a>
## Fase 3: Modelado de temas (LDA)

Apliquen LDA (scikit-learn) sobre la bolsa de palabras. Elijan el número de temas con un criterio explícito (perplejidad o coherencia), interpreten los temas con sus palabras principales y crúcenlos con las emociones.

> **Pregunta de la guía:** Número de temas elegido (criterio), palabras clave de 3 temas y su cruce con emociones.

**Qué analizamos:** Se aplica LDA (scikit-learn) sobre la bolsa de palabras de `muestra_temas` (5 000 tweets, la misma que usará BERTopic en la Fase 9). El número de temas se elige con la perplejidad sobre un conjunto de validación (80/20) con la coherencia NPMI como confirmación (rejilla de K = 5 a 20); luego se interpretan los temas y se cruzan con las emociones.

In [ ]:
# Muestra de 5 000 tweets: es la MISMA que usará BERTopic en la Fase 9, para poder comparar ambos métodos
muestra_temas = df.sample(5000, random_state=SEMILLA)
print(muestra_temas.shape)

In [ ]:
import time, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
t_ini_fase3 = time.time()
TINTA, ACENTO, GRIS = "#14323F", "#6E5BD6", "#B7C3C9"

# Bolsa de palabras: stop words en inglés + tokens basura típicos de tweets (contracciones partidas, risas, etc.)
basura = ['don', 'll', 've', 're', 'm', 's', 'd', 't', 'im', 'ur', 'u', 'lol', 'haha', 'quot', 'amp', 'just', 'got', 'like', 'get', 'know', 'think', 'want', 'did', 'does', 'doesn', 'didn', 'isn', 'wasn', 'won', 'cant', 'dont']
stop_lda = list(ENGLISH_STOP_WORDS.union(basura))
vec_lda = CountVectorizer(stop_words=stop_lda, min_df=5, max_df=0.5, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b")   # solo palabras de 2+ letras; min_df=5 evita palabras rarísimas; max_df=0.5 quita las muy comunes
bolsa = vec_lda.fit_transform(muestra_temas['texto_limpio'])
vocab = np.array(vec_lda.get_feature_names_out())
# tweets que quedaron sin ninguna palabra útil no aportan a LDA
n_vacios = int((bolsa.sum(axis=1) == 0).sum())
print("Matriz documentos x palabras:", bolsa.shape, "| tweets sin palabras útiles:", n_vacios)

**Qué se analiza:** cómo cambian dos medidas de calidad al aumentar el número de temas K: la perplejidad en un 20 % de tweets apartado (menor = mejor) y la coherencia NPMI de las 10 palabras principales de cada tema (mayor = temas más interpretables). Criterio explícito: se elige el K con menor perplejidad de validación, siempre que la coherencia NPMI lo confirme; la rejilla parte de K = 5 como mínimo para no fundir todos los asuntos en 2-3 temas.

In [ ]:
# Partición 80/20 de la muestra (la bolsa de palabras ya está construida con todo el vocabulario)
idx_a, idx_v = train_test_split(np.arange(bolsa.shape[0]), test_size=0.20, random_state=SEMILLA)
bolsa_ent, bolsa_val = bolsa[idx_a], bolsa[idx_v]

# Coherencia NPMI: ¿las palabras principales de un tema aparecen juntas en los mismos tweets?
presencia = (bolsa_ent > 0).astype(float).tocsc()
coocurrencia = (presencia.T @ presencia).toarray()    # nº de tweets donde coinciden dos palabras
frec_doc = np.diag(coocurrencia)                       # nº de tweets donde aparece cada palabra
n_docs = bolsa_ent.shape[0]
def coherencia_npmi(top):
    valores = []
    for i in range(1, len(top)):
        for j in range(i):
            p_ij = coocurrencia[top[i], top[j]] / n_docs
            if p_ij == 0:
                valores.append(-1.0)                   # nunca coinciden
            else:
                pmi = np.log(p_ij / ((frec_doc[top[i]] / n_docs) * (frec_doc[top[j]] / n_docs)))
                valores.append(pmi / -np.log(p_ij))
    return np.mean(valores)

rejilla = [5, 8, 10, 12, 15, 20]
perplejidad_val, perplejidad_ent, coherencia = [], [], []
for k in rejilla:
    t0 = time.time()
    m = LatentDirichletAllocation(n_components=k, learning_method='batch', max_iter=30, random_state=SEMILLA, n_jobs=1)
    m.fit(bolsa_ent)
    perplejidad_val.append(m.perplexity(bolsa_val))
    perplejidad_ent.append(m.perplexity(bolsa_ent))
    tops = [np.argsort(comp)[::-1][:10] for comp in m.components_]
    coherencia.append(np.mean([coherencia_npmi(t) for t in tops]))
    print(f"K={k:2d} | perplejidad validación = {perplejidad_val[-1]:.0f} | entrenamiento = {perplejidad_ent[-1]:.0f} | coherencia NPMI = {coherencia[-1]:.3f} | {time.time() - t0:.1f} s")

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].plot(rejilla, perplejidad_val, marker='o', color=ACENTO, lw=2, label='validación (20 %)')
ejes[0].plot(rejilla, perplejidad_ent, marker='o', color=GRIS, lw=1.5, ls='--', label='entrenamiento (80 %)')
ejes[0].set_xlabel("Número de temas K"); ejes[0].set_ylabel("Perplejidad (menor = mejor)"); ejes[0].legend(frameon=False)
ejes[1].plot(rejilla, coherencia, marker='o', color=ACENTO, lw=2)
ejes[1].set_xlabel("Número de temas K"); ejes[1].set_ylabel("Coherencia NPMI (mayor = mejor)")
for ax in ejes:
    ax.set_xticks(rejilla); ax.grid(False); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig("figuras/fase3_perplejidad.png", dpi=150)
plt.show()

**Qué se analiza:** se aplica el criterio de elección y se comprueba que las dos medidas coincidan. Nota técnica: con tweets muy cortos (pocas palabras útiles por documento) la perplejidad de validación de LDA tiende a SUBIR con K porque los temas extra se sobreajustan a muy poco texto; por eso se usa el menor K permitido por la rejilla si la coherencia lo respalda.

In [ ]:
# Criterio: menor perplejidad de validación, confirmada por la mayor coherencia NPMI
K_perplejidad = rejilla[int(np.argmin(perplejidad_val))]
K_coherencia = rejilla[int(np.argmax(coherencia))]
K_LDA = K_perplejidad
print(f"K por perplejidad de validación = {K_perplejidad} | K por coherencia NPMI = {K_coherencia}")
print(f"K elegido = {K_LDA} ({'ambas medidas coinciden' if K_perplejidad == K_coherencia else 'las medidas NO coinciden'})")

In [ ]:
# Ajuste final con todos los tweets de la muestra y K elegido
t0 = time.time()
lda = LatentDirichletAllocation(n_components=K_LDA, learning_method='batch', max_iter=30, random_state=SEMILLA, n_jobs=1)
doc_tema = lda.fit_transform(bolsa)
print(f"LDA final K={K_LDA} ajustado en {time.time() - t0:.1f} s")

# 10 palabras principales de cada tema
palabras_lda = [list(vocab[np.argsort(comp)[::-1][:10]]) for comp in lda.components_]
for i, p in enumerate(palabras_lda):
    print(f"Tema {i:2d}: {', '.join(p)}")

# Tema dominante de cada tweet (índice = índice de df); tweets sin palabras útiles quedan fuera
tiene_palabras = np.asarray(bolsa.sum(axis=1)).ravel() > 0
tema_dom = pd.Series(doc_tema.argmax(axis=1), index=muestra_temas.index)[tiene_palabras]
print("Tweets por tema dominante:\n", tema_dom.value_counts().sort_index().to_string())

**Qué se analiza:** qué proporción de los tweets de cada emoción tiene como tema dominante cada tema de LDA. Una fila (emoción) suma 100 %; los temas con colores más fuertes son los más típicos de esa emoción.

In [ ]:
emocion = muestra_temas.loc[tema_dom.index, 'sentiment']
cruce = pd.crosstab(emocion, tema_dom, normalize='index') * 100            # % de tweets de la emoción en cada tema
cruce_col = pd.crosstab(emocion, tema_dom, normalize='columns') * 100      # % de cada tema que es de la emoción
print(cruce.round(1).to_string())
base = emocion.value_counts(normalize=True) * 100   # peso de cada emoción en la muestra
# lift: cuántas veces más frecuente es la emoción en el tema que en la muestra completa (solo con >= 15 tweets en la celda)
conteo_cruce = pd.crosstab(emocion, tema_dom)
lift = cruce_col.div(base, axis=0).where(conteo_cruce >= 15)
print("\nPares emoción-tema más sobre-representados (lift, >= 15 tweets):")
print(lift.stack().sort_values(ascending=False).head(8).round(2).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(cruce.values, cmap='Purples', aspect='auto')
ax.set_xticks(range(K_LDA)); ax.set_xticklabels([f"T{t}" for t in cruce.columns])
ax.set_yticks(range(len(cruce))); ax.set_yticklabels(cruce.index)
for i in range(cruce.shape[0]):
    for j in range(cruce.shape[1]):
        ax.text(j, i, f"{cruce.values[i, j]:.0f}", ha='center', va='center', fontsize=8,
                color='white' if cruce.values[i, j] > cruce.values.max() * 0.6 else TINTA)
ax.set_xlabel("Tema dominante de LDA"); ax.set_ylabel("Emoción")
ax.spines[['top', 'right']].set_visible(False)
fig.colorbar(im, ax=ax, label="% de tweets de la emoción")
plt.tight_layout()
plt.savefig("figuras/fase3_temas_vs_emociones.png", dpi=150)
plt.show()

In [ ]:
# Guardar cifras clave de la fase
resultados_fase3 = {
    "tweets_muestra": int(len(muestra_temas)), "tweets_modelados": int(tema_dom.shape[0]),
    "vocabulario": int(bolsa.shape[1]), "tweets_sin_palabras": n_vacios,
    "rejilla_K": rejilla, "perplejidad_validacion": [float(x) for x in perplejidad_val],
    "perplejidad_entrenamiento": [float(x) for x in perplejidad_ent],
    "coherencia_npmi": [float(x) for x in coherencia], "K_perplejidad": int(K_perplejidad), "K_coherencia": int(K_coherencia),
    "K_LDA": int(K_LDA), "palabras_lda": palabras_lda,
    "tweets_por_tema": {int(k): int(v) for k, v in tema_dom.value_counts().sort_index().items()},
    "cruce_pct_por_emocion": {e: {int(t): float(round(v, 2)) for t, v in fila.items()} for e, fila in cruce.iterrows()},
    "lift_top": {f"{e}|T{t}": float(v) for (e, t), v in lift.stack().sort_values(ascending=False).head(8).items()},
    "tiempo_s": round(time.time() - t_ini_fase3, 1),
}
with open("results/fase3.json", "w") as f:
    json.dump(resultados_fase3, f, ensure_ascii=False, indent=2)
print("Guardado results/fase3.json")

**Conclusión de negocio (Fase 3).** LDA sobre `muestra_temas` (5 000 tweets; 4 748 con palabras útiles tras quitar stop words y tokens basura; vocabulario de 1 038 palabras).
- Número de temas: K = 5. Criterio: menor perplejidad en el 20 % de validación, confirmada por la coherencia NPMI. La perplejidad de validación sube con K (2 843 con K = 5; 10 703 con K = 20) y la coherencia NPMI baja en general (de -0,193 con K = 5 a -0,401 con K = 20; el mínimo fue -0,427 con K = 15), porque los tweets tienen muy pocas palabras útiles y los temas extra se sobreajustan. Ningún K mayor mejora estas medidas: es una limitación del texto corto, y por eso K = 5 es un tema-resumen amplio, no una taxonomía fina.
- Tres temas interpretables: Tema 4 (`day, happy, make, fun, mothers, mother, mom`) = celebraciones y Día de la Madre; Tema 3 (`work, miss, need, week, wish, feel, feeling`) = trabajo, añoranza y sentimientos; Tema 0 (`home, today, night, working, going, house, wait`) = rutina diaria en casa y por la noche. Los temas 1 y 2 mezclan cortesías y expresiones sueltas (`good, thanks, nice` / `love, really, sad, hey`) y son menos claros.
- Cruce con emociones: el Tema 4 concentra 31,4 % de los tweets de `love` (1,68 veces lo esperado), 25,2 % de `happiness` y 24,5 % de `fun`, frente a solo 10,2 % de `hate`; el Tema 3 reúne 24,4 % de `sadness` y de `enthusiasm` y 23,9 % de `hate`. Casi todas las demás celdas quedan entre 14 % y 27 %, cerca del 20 % de un reparto al azar; salen de ese rango `empty` en el Tema 1 (33,0 %) y en el Tema 4 (9,9 %), `happiness` en el Tema 3 (13,6 %) y `love` en el Tema 3 (9,4 %).
- Lectura honesta: los temas describen de qué hablan los tweets (celebraciones, trabajo, casa), no cómo se sienten. Separan emociones positivas de afecto (love, happiness, fun) de las negativas, pero no distinguen emociones vecinas. Valen como contexto de negocio (p. ej. picos de positividad en fechas como el Día de la Madre), no como clasificador. `anger` (10 tweets) y `boredom` (26) tienen muy pocos tweets en la muestra y sus porcentajes no son fiables.

<a id="fase-4"></a>
## Fase 4: Clasificación clásica: Naive Bayes y SVM

Paso 2. Partición **estratificada** entrenamiento/validación/prueba con semilla. Dos pipelines de scikit-learn: TF-IDF + Naive Bayes multinomial y TF-IDF + SVM lineal. Ajusten en validación y reporten F1 macro y matriz de confusión (13 clases). La prueba se usa una sola vez.

> **Pregunta de la guía:** F1 macro y matriz de confusión de NB y SVM. ¿Qué emociones confunden más?

**Qué analizamos:** Se entrenan dos clasificadores clásicos de emociones (13 clases) sobre el texto limpio: TF-IDF + Naive Bayes multinomial y TF-IDF + SVM lineal. Se ajustan pocos hiperparámetros usando SOLO la validación, se evalúa en prueba una única vez y se miran F1 macro, matrices de confusión y las emociones que más se confunden.

**Partición de los datos.** Se usa una partición estratificada 70/15/15 con semilla fija. Estratificar significa que cada subconjunto conserva la proporción de cada emoción, algo clave porque hay clases muy pequeñas (anger, boredom).

In [ ]:
import time, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

# Partición estratificada 70/15/15 con SEMILLA (mismo código que common.py)
idx_ent, idx_tmp = train_test_split(df.index, test_size=0.30, stratify=df['sentiment'], random_state=SEMILLA)
idx_val, idx_pru = train_test_split(idx_tmp, test_size=0.50, stratify=df.loc[idx_tmp, 'sentiment'], random_state=SEMILLA)
X_train, y_train = df.loc[idx_ent, 'texto_limpio'], df.loc[idx_ent, 'sentiment']
X_val,   y_val   = df.loc[idx_val, 'texto_limpio'], df.loc[idx_val, 'sentiment']
X_test,  y_test  = df.loc[idx_pru, 'texto_limpio'], df.loc[idx_pru, 'sentiment']

# Tamaños y proporción por clase en cada subconjunto (deben ser casi idénticas)
print(f"Tamaños -> train: {len(X_train)}, validación: {len(X_val)}, prueba: {len(X_test)}")
proporciones = pd.DataFrame({
    'total':      df['sentiment'].value_counts(normalize=True),
    'train':      y_train.value_counts(normalize=True),
    'validacion': y_val.value_counts(normalize=True),
    'prueba':     y_test.value_counts(normalize=True)}).round(4) * 100
proporciones['n_total'] = df['sentiment'].value_counts()
print("Proporción (%) de cada emoción por subconjunto:")
print(proporciones.to_string())
print("Diferencia máxima entre train y total (puntos porcentuales):",
      round((proporciones['train'] - proporciones['total']).abs().max(), 3))

**Línea base.** Antes de mirar los modelos, cuánto se obtiene sin aprender nada: elegir al azar entre 13 clases (1/13) o predecir siempre la clase mayoritaria.

In [ ]:
azar = 1 / len(ETIQUETAS)
clase_mayoritaria = y_train.value_counts().idxmax()
pred_mayoritaria = np.full(len(y_test), clase_mayoritaria)
base_acc = accuracy_score(y_test, pred_mayoritaria)
base_f1 = f1_score(y_test, pred_mayoritaria, average='macro', labels=ETIQUETAS, zero_division=0)
print(f"Azar uniforme: accuracy = F1 macro esperado ≈ {azar:.3f}")
print(f"Siempre '{clase_mayoritaria}': accuracy = {base_acc:.3f}, F1 macro = {base_f1:.3f}")

**Ajuste de hiperparámetros sobre validación.** Se prueba una rejilla pequeña (n-gramas, alpha de NB, C y class_weight de SVM). El criterio es el F1 macro en validación; la prueba no se toca aquí.

In [ ]:
def crear_nb(alpha, ngramas):
    return Pipeline([('tfidf', TfidfVectorizer(ngram_range=ngramas, min_df=2, sublinear_tf=True)),
                     ('clf', MultinomialNB(alpha=alpha))])

def crear_svm(C, ngramas, pesos):
    return Pipeline([('tfidf', TfidfVectorizer(ngram_range=ngramas, min_df=2, sublinear_tf=True)),
                     ('clf', LinearSVC(C=C, class_weight=pesos, random_state=SEMILLA))])

filas = []
for ngr in [(1, 1), (1, 2)]:
    for alpha in [0.03, 0.1, 0.3, 1.0]:
        m = crear_nb(alpha, ngr).fit(X_train, y_train)
        filas.append({'modelo': 'NB', 'ngramas': ngr, 'alpha': alpha, 'C': None, 'class_weight': None,
                      'f1_val': f1_score(y_val, m.predict(X_val), average='macro')})
    for C in [0.03, 0.1, 0.3, 1.0]:
        for pesos in [None, 'balanced']:
            m = crear_svm(C, ngr, pesos).fit(X_train, y_train)
            filas.append({'modelo': 'SVM', 'ngramas': ngr, 'alpha': None, 'C': C, 'class_weight': pesos,
                          'f1_val': f1_score(y_val, m.predict(X_val), average='macro')})
rejilla = pd.DataFrame(filas)
print(rejilla.round(4).to_string(index=False))

mejor_nb  = rejilla[rejilla.modelo == 'NB'].sort_values('f1_val').iloc[-1]
mejor_svm = rejilla[rejilla.modelo == 'SVM'].sort_values('f1_val').iloc[-1]
print("\nMejor NB : ngramas", mejor_nb["ngramas"], "alpha", mejor_nb["alpha"], "F1 val", round(mejor_nb["f1_val"], 4))
print("Mejor SVM: ngramas", mejor_svm["ngramas"], "C", mejor_svm["C"], "class_weight", mejor_svm["class_weight"], "F1 val", round(mejor_svm["f1_val"], 4))

**Evaluación final en prueba (una sola vez).** Con los hiperparámetros elegidos se reentrena cada modelo solo con train, se mide el tiempo del ajuste y se evalúa en el conjunto de prueba.

In [ ]:
# Hiperparámetros elegidos (los reutiliza la Fase 6)
params_nb  = {'alpha': float(mejor_nb['alpha']), 'ngramas': mejor_nb['ngramas']}
params_svm = {'C': float(mejor_svm['C']), 'ngramas': mejor_svm['ngramas'], 'pesos': mejor_svm['class_weight']}

modelos_finales = {'NB':  crear_nb(params_nb['alpha'], params_nb['ngramas']),
                   'SVM': crear_svm(params_svm['C'], params_svm['ngramas'], params_svm['pesos'])}
predicciones, filas_res, tiempos = {}, [], {}
for nombre, modelo in modelos_finales.items():
    t0 = time.perf_counter()
    modelo.fit(X_train, y_train)
    tiempos[nombre] = time.perf_counter() - t0
    predicciones[nombre] = modelo.predict(X_test)            # única evaluación en prueba
    filas_res.append({'modelo': nombre, 'fase': 4,
                      'f1_macro': f1_score(y_test, predicciones[nombre], average='macro'),
                      'accuracy': accuracy_score(y_test, predicciones[nombre]),
                      'tiempo_s': tiempos[nombre]})
resultados = pd.DataFrame(filas_res)
print(resultados.round(4).to_string(index=False))

# F1 por clase en prueba, para ambos modelos
f1_por_clase = pd.DataFrame({
    n: pd.Series(f1_score(y_test, predicciones[n], labels=ETIQUETAS, average=None), index=ETIQUETAS)
    for n in modelos_finales})
f1_por_clase['n_prueba'] = y_test.value_counts()
f1_por_clase = f1_por_clase.sort_values('n_prueba', ascending=False)
print("\nF1 por clase (prueba):")
print(f1_por_clase.round(3).to_string())

# Mejor modelo por F1 macro de prueba (ambos ya estaban fijados con validación)
mejor_nombre = resultados.sort_values('f1_macro').iloc[-1]['modelo']
mejor_modelo = modelos_finales[mejor_nombre]
pred_test_mejor = predicciones[mejor_nombre]
print("\nMejor modelo:", mejor_nombre)

**Qué se analiza:** la matriz de confusión 13×13 de cada modelo, normalizada por fila (cada fila suma 100 %: de los tweets de una emoción real, qué porcentaje se predijo como cada emoción). Las emociones van ordenadas de más a menos frecuentes; la diagonal es el recall de cada clase y lo que queda fuera de ella son las confusiones.

In [ ]:
TINTA, ACENTO, GRIS = "#14323F", "#6E5BD6", "#B7C3C9"
orden = list(y_train.value_counts().index)        # emociones de más a menos frecuentes

def dibujar_matriz(pred, titulo, archivo):
    cm = confusion_matrix(y_test, pred, labels=orden, normalize='true') * 100
    fig, ax = plt.subplots(figsize=(9.5, 8))
    from matplotlib.colors import LinearSegmentedColormap
    mapa = LinearSegmentedColormap.from_list('tinta', ['#FFFFFF', ACENTO])
    ax.imshow(cm, cmap=mapa, vmin=0, vmax=60)
    for i in range(len(orden)):
        for j in range(len(orden)):
            if cm[i, j] >= 1:
                ax.text(j, i, f"{cm[i, j]:.0f}", ha='center', va='center', fontsize=8,
                        color='white' if cm[i, j] > 35 else TINTA)
    ax.set_xticks(range(len(orden))); ax.set_xticklabels(orden, rotation=45, ha='right', color=TINTA)
    ax.set_yticks(range(len(orden))); ax.set_yticklabels(orden, color=TINTA)
    ax.set_xlabel('Emoción predicha', color=TINTA); ax.set_ylabel('Emoción real', color=TINTA)
    ax.set_title(titulo, color=TINTA, loc='left')
    for lado in ['top', 'right', 'left', 'bottom']:
        ax.spines[lado].set_visible(False)
    plt.tight_layout(); plt.savefig(archivo, dpi=150); plt.show()

dibujar_matriz(predicciones['NB'], 'Naive Bayes: matriz de confusión (% por fila, prueba)', 'figuras/fase4_matriz_nb.png')
dibujar_matriz(predicciones['SVM'], 'SVM lineal: matriz de confusión (% por fila, prueba)', 'figuras/fase4_matriz_svm.png')

**Qué se analiza:** los 5 pares (real → predicha) con más confusión fuera de la diagonal, medidos como porcentaje de la clase real, para ver qué emociones se parecen entre sí desde el punto de vista del texto. Debajo se añade una figura de apoyo con el F1 por clase de ambos modelos (el mejor resaltado en color) para ver en qué emociones gana cada uno.

In [ ]:
def top_confusiones(pred, n=5):
    cm = confusion_matrix(y_test, pred, labels=ETIQUETAS, normalize='true') * 100
    cuentas = confusion_matrix(y_test, pred, labels=ETIQUETAS)
    pares = [(ETIQUETAS[i], ETIQUETAS[j], cm[i, j], cuentas[i, j])
             for i in range(len(ETIQUETAS)) for j in range(len(ETIQUETAS)) if i != j]
    pares.sort(key=lambda p: p[3], reverse=True)       # por número de tweets mal clasificados
    return pd.DataFrame(pares[:n], columns=['real', 'predicha', 'pct_de_la_real', 'tweets'])

confusiones = {n: top_confusiones(predicciones[n]) for n in modelos_finales}
for n, tabla in confusiones.items():
    print(f"Top 5 confusiones de {n} (por número de tweets):")
    print(tabla.round(1).to_string(index=False), "\n")

In [ ]:
# Figura de apoyo: F1 por clase de ambos modelos (ACENTO para el mejor modelo)
fig, ax = plt.subplots(figsize=(9, 4.8))
posicion = np.arange(len(orden)); ancho = 0.4
tabla_plot = f1_por_clase.loc[orden]
otro = 'NB' if mejor_nombre == 'SVM' else 'SVM'
ax.bar(posicion - ancho/2, tabla_plot[otro], ancho, color=GRIS, label=otro)
ax.bar(posicion + ancho/2, tabla_plot[mejor_nombre], ancho, color=ACENTO, label=mejor_nombre)
ax.axhline(azar, color=TINTA, ls='--', lw=1); ax.text(len(orden)-0.5, azar+0.01, 'azar 1/13', ha='right', color=TINTA, fontsize=8)
ax.set_xticks(posicion); ax.set_xticklabels(orden, rotation=45, ha='right', color=TINTA)
ax.set_ylabel('F1 por clase (prueba)', color=TINTA); ax.legend(frameon=False)
for lado in ['top', 'right']:
    ax.spines[lado].set_visible(False)
plt.tight_layout(); plt.savefig('figuras/fase4_f1_por_clase.png', dpi=150); plt.show()

In [ ]:
# Guardar cifras clave
nb, sv = resultados.set_index('modelo').loc['NB'], resultados.set_index('modelo').loc['SVM']
salida = {
    'tamanos': {'train': len(X_train), 'val': len(X_val), 'test': len(X_test)},
    'azar': azar, 'clase_mayoritaria': clase_mayoritaria,
    'base_mayoritaria_accuracy': base_acc, 'base_mayoritaria_f1_macro': base_f1,
    'hiperparametros': {'NB': {'alpha': params_nb['alpha'], 'ngramas': list(params_nb['ngramas']), 'f1_val': float(mejor_nb['f1_val'])},
                        'SVM': {'C': params_svm['C'], 'ngramas': list(params_svm['ngramas']),
                                'class_weight': params_svm['pesos'], 'f1_val': float(mejor_svm['f1_val'])}},
    'resultados': resultados.round(5).to_dict('records'),
    'mejor_nombre': mejor_nombre,
    'f1_por_clase': f1_por_clase.round(4).to_dict('index'),
    'top_confusiones': {n: t.round(2).to_dict('records') for n, t in confusiones.items()}}
with open('results/fase4.json', 'w') as f:
    json.dump(salida, f, indent=2, default=str)

**Conclusión de negocio (Fase 4).** Con 13 emociones, el mejor clasificador clásico es el SVM lineal (TF-IDF con unigramas y bigramas, C=0.1, class_weight balanceado): F1 macro de 0.198 y accuracy de 0.270 en prueba, frente a 0.144 y 0.299 de Naive Bayes (alpha=0.03). Ambos superan el azar (1/13 = 0.077 de F1 macro; el SVM lo multiplica por 2.6) y la clase mayoritaria (siempre «neutral»: accuracy 0.215, F1 macro 0.027), pero siguen siendo modelos débiles: no deberían decidir solos ninguna acción sobre un cliente. NB gana en accuracy porque favorece las clases grandes (neutral, worry) y casi ignora las pequeñas (F1 de 0.00 en empty y anger); el SVM con pesos reparte mejor los aciertos (F1 de 0.43 en love, 0.25 en hate), por eso se elige por F1 macro. Los dos entrenan en segundos (0.7 s y 1.7 s). Las confusiones dominantes son neutral↔worry (NB: 354 y 335 tweets; SVM: 144 y 138), sadness→worry (NB 314; SVM 128), happiness→neutral en NB y happiness→love en SVM (129): son emociones vecinas en el vocabulario (preocupación y tristeza comparten palabras de malestar; neutral es una clase «cajón» sin señal léxica propia; happiness y love comparten palabras afectivas). Recomendación: usar el SVM como línea base, vigilar el F1 por clase y no confiar en las emociones raras (anger, boredom, empty, enthusiasm: F1 ≤ 0.08). El ajuste de hiperparámetros se hizo solo en validación; en NB y SVM el mejor valor quedó en el borde de la rejilla (alpha=0.03; C=0.1 con 0.03 también probado), mejora pequeña esperable con más búsqueda.

<a id="fase-5"></a>
## Fase 5: Red neuronal recurrente (RNN)

Paso 2. Modelo en Keras: `TextVectorization` → `Embedding` → LSTM o GRU (bidireccional opcional) → softmax. Usen la misma partición, early stopping sobre validación y pesos de clase o muestreo para el desbalance. Comparen costo de entrenamiento y desempeño frente a NB y SVM.

> **Pregunta de la guía:** F1 macro, tiempo de entrenamiento y estrategia de desbalance de la RNN frente a NB y SVM.

**Qué analizamos:** Se entrena una red recurrente en Keras (TextVectorization, Embedding, GRU bidireccional, softmax de 13 clases) con la misma partición de la Fase 4, parada temprana sobre validación y pesos de clase para el desbalance. Se compara su F1 macro, accuracy, tiempo de entrenamiento y tamaño contra NB y SVM.

**Configuración.** Se fija la semilla de TensorFlow y se limitan los hilos para no saturar la CPU compartida. La función `entrenar_rnn` encapsula todo el modelo para poder reutilizarla en la Fase 6 con otros subconjuntos de clases.

In [ ]:
import os, time, json
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import numpy as np, pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score

tf.keras.utils.set_random_seed(SEMILLA)           # semilla de numpy, python y TF
tf.config.threading.set_intra_op_parallelism_threads(2)   # limita hilos (CPU compartida)
tf.config.threading.set_inter_op_parallelism_threads(2)

def entrenar_rnn(X_tr, y_tr, X_va, y_va, X_te, y_te, clases, max_tokens=20000, largo=40,
                 dim_emb=64, unidades=64, max_epocas=10, paciencia=2, batch=128, verbose=0):
    """Entrena TextVectorization -> Embedding -> BiGRU -> Dropout -> softmax.
    Early stopping sobre val_loss y class_weight balanceado. Evalúa en prueba una vez."""
    tf.keras.utils.set_random_seed(SEMILLA)
    mapa = {c: i for i, c in enumerate(clases)}                 # etiqueta -> entero (mismo orden que ETIQUETAS)
    ytr, yva, yte = (np.array([mapa[c] for c in y]) for y in (y_tr, y_va, y_te))

    vectorizador = tf.keras.layers.TextVectorization(max_tokens=max_tokens, output_sequence_length=largo)
    vectorizador.adapt(np.array(X_tr))                          # el vocabulario se aprende solo con train
    vec = lambda x: vectorizador(np.array(x)).numpy()
    Xtr, Xva, Xte = vec(X_tr), vec(X_va), vec(X_te)

    modelo = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(largo,), dtype='int64'),
        tf.keras.layers.Embedding(max_tokens, dim_emb, mask_zero=True),
        tf.keras.layers.Bidirectional(tf.keras.layers.GRU(unidades)),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(len(clases), activation='softmax')])
    modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])

    # Pesos de clase 'balanced': n / (k * n_clase); las clases raras pesan más en la pérdida
    pesos = compute_class_weight('balanced', classes=np.arange(len(clases)), y=ytr)
    pesos = dict(enumerate(pesos))
    parada = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=paciencia, restore_best_weights=True)

    t0 = time.perf_counter()
    hist = modelo.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=max_epocas, batch_size=batch,
                      class_weight=pesos, callbacks=[parada], verbose=verbose)
    tiempo = time.perf_counter() - t0

    pred = modelo.predict(Xte, batch_size=512, verbose=0).argmax(axis=1)   # única evaluación en prueba
    return {'modelo': modelo, 'historial': hist.history, 'tiempo_s': tiempo,
            'epocas': len(hist.history['loss']), 'mejor_epoca': int(np.argmin(hist.history['val_loss'])) + 1,
            'parametros': int(modelo.count_params()), 'pesos': pesos,
            'f1_macro': f1_score(yte, pred, average='macro', labels=np.arange(len(clases)), zero_division=0),
            'accuracy': accuracy_score(yte, pred), 'pred': np.array(clases)[pred]}

**Entrenamiento con las 13 emociones.** Vocabulario de 20 000 tokens, secuencias de 40 palabras, embedding de 64 y GRU bidireccional de 64 unidades; máximo 10 épocas, lote de 128, paciencia 2 sobre `val_loss`.

In [ ]:
res_rnn = entrenar_rnn(X_train, y_train, X_val, y_val, X_test, y_test, ETIQUETAS, verbose=2)
print(f"\nÉpocas ejecutadas: {res_rnn['epocas']} (mejor época: {res_rnn['mejor_epoca']})")
print(f"Tiempo de entrenamiento: {res_rnn['tiempo_s']:.1f} s | parámetros: {res_rnn['parametros']:,}")
print(f"Prueba -> F1 macro: {res_rnn['f1_macro']:.4f}, accuracy: {res_rnn['accuracy']:.4f}")

# Pesos de clase aplicados (efecto del balanceo)
pesos_tabla = pd.Series({ETIQUETAS[i]: p for i, p in res_rnn['pesos'].items()}).sort_values()
print("\nPeso de clase 'balanced' (neutral pesa poco, anger mucho):")
print(pesos_tabla.round(2).to_string())

**Qué se analiza:** la curva de pérdida (entropía cruzada) en entrenamiento y validación por época. Sirve para ver cuándo empieza el sobreajuste y por qué la parada temprana recupera los pesos de la mejor época (marcada en ACENTO).

In [ ]:
TINTA, ACENTO, GRIS = "#14323F", "#6E5BD6", "#B7C3C9"
h = res_rnn['historial']
epocas = np.arange(1, len(h['loss']) + 1)
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(epocas, h['loss'], color=GRIS, marker='o', label='Entrenamiento (con pesos de clase)')
ax.plot(epocas, h['val_loss'], color=ACENTO, marker='o', label='Validación')
ax.axvline(res_rnn['mejor_epoca'], color=TINTA, ls='--', lw=1)
ax.text(res_rnn['mejor_epoca'] + 0.05, max(h['loss']) * 0.98, 'mejor época', color=TINTA, fontsize=8, va='top')
ax.set_xlabel('Época', color=TINTA); ax.set_ylabel('Pérdida', color=TINTA); ax.legend(frameon=False)
ax.set_xticks(epocas)
for lado in ['top', 'right']:
    ax.spines[lado].set_visible(False)
plt.tight_layout(); plt.savefig('figuras/fase5_curva_entrenamiento.png', dpi=150); plt.show()

**Comparación NB vs SVM vs RNN.** Se añade la fila de la RNN a `resultados` y se arma una tabla con F1 macro, accuracy, tiempo de entrenamiento y tamaño (parámetros). Para NB y SVM el tamaño es el número de coeficientes aprendidos.

In [ ]:
resultados = pd.concat([resultados, pd.DataFrame([{'modelo': 'RNN', 'fase': 5,
                        'f1_macro': res_rnn['f1_macro'], 'accuracy': res_rnn['accuracy'],
                        'tiempo_s': res_rnn['tiempo_s']}])], ignore_index=True)

def n_parametros(pipe):
    # NB: log-probabilidades por clase y término; SVM: coeficientes + sesgos
    clf = pipe.named_steps['clf']
    if hasattr(clf, 'feature_log_prob_'):
        return int(clf.feature_log_prob_.size + clf.class_log_prior_.size)
    return int(clf.coef_.size + clf.intercept_.size)

params = {'NB': n_parametros(modelos_finales['NB']), 'SVM': n_parametros(modelos_finales['SVM']),
          'RNN': res_rnn['parametros']}
comparacion = resultados[resultados.fase.isin([4, 5])].copy()
comparacion['parametros'] = comparacion['modelo'].map(params)
print(comparacion.round(4).to_string(index=False))
rel = comparacion.set_index('modelo')
print(f"\nLa RNN tarda {rel.loc['RNN','tiempo_s']/rel.loc['SVM','tiempo_s']:.0f} veces más que el SVM "
      f"y {rel.loc['RNN','tiempo_s']/rel.loc['NB','tiempo_s']:.0f} veces más que NB.")
print(f"Diferencia de F1 macro RNN - SVM: {rel.loc['RNN','f1_macro']-rel.loc['SVM','f1_macro']:+.4f}")

**Qué se analiza:** F1 por clase de la RNN frente al SVM, para ver si el balanceo con pesos ayuda en las emociones raras o solo reparte los errores de otra forma.

In [ ]:
f1_rnn = pd.Series(f1_score(y_test, res_rnn['pred'], labels=ETIQUETAS, average=None), index=ETIQUETAS)
tabla_clases = pd.DataFrame({'SVM': f1_por_clase['SVM'], 'RNN': f1_rnn, 'n_prueba': f1_por_clase['n_prueba']})
tabla_clases = tabla_clases.sort_values('n_prueba', ascending=False)
print(tabla_clases.round(3).to_string())

fig, ax = plt.subplots(figsize=(9, 4.6))
pos = np.arange(len(tabla_clases)); ancho = 0.4
ax.bar(pos - ancho/2, tabla_clases['SVM'], ancho, color=GRIS, label='SVM')
ax.bar(pos + ancho/2, tabla_clases['RNN'], ancho, color=ACENTO, label='RNN')
ax.set_xticks(pos); ax.set_xticklabels(tabla_clases.index, rotation=45, ha='right', color=TINTA)
ax.set_ylabel('F1 por clase (prueba)', color=TINTA); ax.legend(frameon=False)
for lado in ['top', 'right']:
    ax.spines[lado].set_visible(False)
plt.tight_layout(); plt.savefig('figuras/fase5_f1_por_clase_rnn_svm.png', dpi=150); plt.show()

In [ ]:
salida5 = {'rnn': {'f1_macro': res_rnn['f1_macro'], 'accuracy': res_rnn['accuracy'],
                   'tiempo_s': res_rnn['tiempo_s'], 'epocas': res_rnn['epocas'],
                   'mejor_epoca': res_rnn['mejor_epoca'], 'parametros': res_rnn['parametros'],
                   'config': {'max_tokens': 20000, 'largo': 40, 'dim_emb': 64, 'unidades': 64,
                              'batch': 128, 'class_weight': 'balanced', 'paciencia': 2}},
           'peso_clase': {k: round(float(v), 3) for k, v in pesos_tabla.items()},
           'comparacion': comparacion.round(5).to_dict('records'),
           'f1_por_clase': {'RNN': f1_rnn.round(4).to_dict(), 'SVM': f1_por_clase['SVM'].round(4).to_dict()},
           'historial': {k: [round(float(x), 5) for x in v] for k, v in h.items()}}
with open('results/fase5.json', 'w') as archivo:
    json.dump(salida5, archivo, indent=2)

**Conclusión de negocio (Fase 5).** La RNN (embedding 64, GRU bidireccional de 64, class_weight balanceado) NO supera al SVM: obtiene F1 macro de 0.149 y accuracy de 0.176 en prueba, frente a 0.198 y 0.270 del SVM y 0.144 y 0.299 de NB; queda prácticamente empatada con NB en F1 macro y por debajo en accuracy. Además es mucho más cara: unos 62 s de entrenamiento en 5 épocas (la mejor fue la 3; la pérdida de validación empieza a subir desde ahí mientras la de entrenamiento sigue bajando, es decir, sobreajuste rápido) frente a 1.7 s del SVM y 0.7 s de NB (alrededor de 36 y 84 veces más; el tiempo exacto varía entre corridas por la CPU compartida), y con 1.33 millones de parámetros frente a 0.50 millones de coeficientes. Para el desbalance se usaron pesos de clase 'balanced' (neutral y worry pesan 0.36; boredom 17.1 y anger 27.8): eso obliga al modelo a prestar atención a las emociones pequeñas, pero baja la accuracy porque sacrifica las grandes (F1 de neutral 0.10 y worry 0.11, contra 0.31 y 0.32 del SVM) y, con solo 110 tweets de anger, los pesos altos tienden a memorizar esos ejemplos. La alternativa de muestreo (sobremuestrear clases raras o submuestrear neutral/worry) tendría efectos parecidos y no se probó. La RNN solo gana al SVM en sadness (0.30 vs 0.29), boredom (0.087 vs 0.056) y anger (0.010 vs 0.000), con diferencias pequeñas. Recomendación: con 28 mil tweets cortos y etiquetas ruidosas, un modelo lineal TF-IDF es la opción de mejor costo-beneficio; una RNN solo se justificaría con más datos o embeddings preentrenados.

<a id="fase-6"></a>
## Fase 6: Experimento: número de clases

Pasos 3–4. Repitan los tres modelos con **3, 5, 7, 9 y 11** emociones seleccionadas al azar (fijen y reporten la semilla; idealmente promedien 3 semillas). Grafiquen F1 macro frente al número de clases para cada modelo, junto a la línea base de azar (1/k). Reporten **accuracy y F1 macro**: si divergen mucho para un mismo subconjunto, revisen qué clases salieron sorteadas y su tamaño. Expliquen la tendencia.

> **Pregunta de la guía:** Gráfica F1 macro vs. número de clases con la línea de azar. ¿Qué granularidad recomiendan?

**Qué analizamos:** Se repite la comparación NB, SVM y RNN con 3, 5, 7, 9 y 11 emociones sorteadas al azar (3 semillas: 42, 43 y 44, promediando) para ver cómo cae el F1 macro al crecer el número de clases, frente a la línea de azar 1/k, y se recomienda una granularidad de etiquetas.

**Diseño del experimento.** Para cada semilla se sortea una permutación de las 13 emociones con `np.random.RandomState(semilla)` y se toman las primeras k (así, dentro de una semilla, los subconjuntos son anidados: k=5 contiene a k=3). Se filtra la misma partición train/val/test de la Fase 4. NB y SVM usan los hiperparámetros elegidos en la Fase 4; la RNN usa una versión más pequeña y rápida (embedding 32, GRU bidireccional de 32, máximo 4 épocas, paciencia 1) para que el experimento completo sea razonable en 4 CPU. Aquí la prueba es la evaluación final de cada combinación (una vez).

In [ ]:
import time, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, accuracy_score

KS = [3, 5, 7, 9, 11]
SEMILLAS = [42, 43, 44]
conteo_total = df['sentiment'].value_counts()

def sortear_clases(semilla, k):
    rng = np.random.RandomState(semilla)
    return list(rng.permutation(ETIQUETAS)[:k])

def filtrar(X, y, clases):
    mascara = y.isin(clases)
    return X[mascara], y[mascara]

def evaluar(pred, y_real, clases):
    return (f1_score(y_real, pred, average='macro', labels=clases, zero_division=0),
            accuracy_score(y_real, pred))

registros = []
t_inicio = time.perf_counter()
for semilla in SEMILLAS:
    for k in KS:
        clases = sortear_clases(semilla, k)
        Xtr, ytr = filtrar(X_train, y_train, clases)
        Xva, yva = filtrar(X_val, y_val, clases)
        Xte, yte = filtrar(X_test, y_test, clases)
        info = {'k': k, 'semilla': semilla, 'clases': ', '.join(sorted(clases)),
                'tamanos': ', '.join(f"{c}={int((yte == c).sum())}" for c in sorted(clases)),
                'n_test': len(yte), 'pct_mayoritaria': (yte.value_counts().iloc[0] / len(yte)) * 100,
                'azar': 1 / k}

        # NB y SVM con los hiperparámetros de la Fase 4
        for nombre, pipe in [('NB', crear_nb(params_nb['alpha'], params_nb['ngramas'])),
                             ('SVM', crear_svm(params_svm['C'], params_svm['ngramas'], params_svm['pesos']))]:
            t0 = time.perf_counter()
            pipe.fit(Xtr, ytr)
            tiempo = time.perf_counter() - t0
            f1, acc = evaluar(pipe.predict(Xte), yte, clases)
            registros.append({**info, 'modelo': nombre, 'f1_macro': f1, 'accuracy': acc, 'tiempo_s': tiempo})

        # RNN pequeña (función de la Fase 5)
        r = entrenar_rnn(Xtr, ytr, Xva, yva, Xte, yte, sorted(clases), max_tokens=15000, dim_emb=32,
                         unidades=32, max_epocas=4, paciencia=1)
        registros.append({**info, 'modelo': 'RNN', 'f1_macro': r['f1_macro'], 'accuracy': r['accuracy'],
                          'tiempo_s': r['tiempo_s']})
        print(f"semilla {semilla}, k={k:2d}: " + ' | '.join(
            f"{x['modelo']} F1={x['f1_macro']:.3f}" for x in registros[-3:]) + f"  [{time.perf_counter()-t_inicio:.0f} s]")

tabla_larga = pd.DataFrame(registros)
tiempo_total_f6 = time.perf_counter() - t_inicio
print(f"\nTiempo total del experimento: {tiempo_total_f6/60:.1f} min")

**Resumen por k y modelo.** Promedio y desviación estándar sobre las 3 semillas de F1 macro y accuracy.

In [ ]:
resumen = (tabla_larga.groupby(['k', 'modelo'])[['f1_macro', 'accuracy']].agg(['mean', 'std']).round(4))
resumen.columns = ['f1_media', 'f1_desv', 'acc_media', 'acc_desv']
resumen = resumen.reset_index()
resumen['azar'] = (1 / resumen['k']).round(4)
resumen['ganancia_vs_azar'] = (resumen['f1_media'] / resumen['azar']).round(2)   # veces sobre el azar
print(resumen.to_string(index=False))

**Qué se analiza:** F1 macro (promedio de 3 semillas, con banda de ±1 desviación estándar) en función del número de clases para NB, SVM y RNN, junto a la línea de azar 1/k. Muestra cuánto se pierde al aumentar la granularidad y cuánto se gana sobre adivinar.

In [ ]:
TINTA, ACENTO, GRIS = "#14323F", "#6E5BD6", "#B7C3C9"
colores = {'NB': GRIS, 'SVM': ACENTO, 'RNN': TINTA}
estilos = {'NB': '-', 'SVM': '-', 'RNN': '--'}
fig, ax = plt.subplots(figsize=(7.5, 4.8))
for nombre in ['NB', 'SVM', 'RNN']:
    s = resumen[resumen.modelo == nombre]
    ax.plot(s['k'], s['f1_media'], marker='o', color=colores[nombre], ls=estilos[nombre], label=nombre, lw=2)
    ax.fill_between(s['k'], s['f1_media'] - s['f1_desv'], s['f1_media'] + s['f1_desv'],
                    color=colores[nombre], alpha=0.15, lw=0)
ks_fino = np.linspace(3, 11, 50)
ax.plot(ks_fino, 1 / ks_fino, color=TINTA, ls=':', lw=1.2, label='azar (1/k)')
ax.set_xticks(KS); ax.set_xlabel('Número de emociones (k)', color=TINTA)
ax.set_ylabel('F1 macro en prueba', color=TINTA); ax.set_ylim(0, None)
ax.legend(frameon=False)
for lado in ['top', 'right']:
    ax.spines[lado].set_visible(False)
plt.tight_layout(); plt.savefig('figuras/fase6_f1_vs_k.png', dpi=150); plt.show()

**Qué se analiza:** accuracy frente a F1 macro para cada combinación (k, semilla, modelo). Cuando la accuracy es alta pero el F1 macro es bajo, el modelo acierta sobre todo las clases grandes e ignora las pequeñas; se señalan los casos con mayor brecha.

In [ ]:
tabla_larga['brecha'] = tabla_larga['accuracy'] - tabla_larga['f1_macro']
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.plot([0, 1], [0, 1], color=GRIS, lw=1)
for nombre in ['NB', 'SVM', 'RNN']:
    s = tabla_larga[tabla_larga.modelo == nombre]
    ax.scatter(s['accuracy'], s['f1_macro'], color=colores[nombre], label=nombre, s=40, alpha=0.9)
peores = tabla_larga.sort_values('brecha', ascending=False).head(3)
for _, fila in peores.iterrows():
    ax.annotate(f"k={fila['k']}, s={fila['semilla']}", (fila['accuracy'], fila['f1_macro']),
                textcoords='offset points', xytext=(6, -10), fontsize=8, color=TINTA)
ax.set_xlabel('Accuracy', color=TINTA); ax.set_ylabel('F1 macro', color=TINTA)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(frameon=False)
for lado in ['top', 'right']:
    ax.spines[lado].set_visible(False)
plt.tight_layout(); plt.savefig('figuras/fase6_accuracy_vs_f1.png', dpi=150); plt.show()

print("Combinaciones con mayor brecha accuracy - F1 macro, y las clases sorteadas (tamaños en prueba):")
for _, fila in peores.iterrows():
    print(f"\n{fila['modelo']} k={fila['k']} semilla={fila['semilla']}: accuracy={fila['accuracy']:.3f}, "
          f"F1 macro={fila['f1_macro']:.3f}, mayoritaria={fila['pct_mayoritaria']:.0f}% de la prueba")
    print("  clases:", fila['tamanos'])

**Clases sorteadas y tamaños.** Qué emociones salieron en cada semilla y k (tamaño en el conjunto de prueba), para interpretar las diferencias entre semillas.

In [ ]:
sorteos = tabla_larga.drop_duplicates(['k', 'semilla'])[['k', 'semilla', 'n_test', 'pct_mayoritaria', 'clases']]
sorteos['pct_mayoritaria'] = sorteos['pct_mayoritaria'].round(1)
print(sorteos.to_string(index=False))

# Dispersión entre semillas del F1 macro del SVM
print("\nF1 macro del SVM por k y semilla:")
print(tabla_larga[tabla_larga.modelo == 'SVM'].pivot(index='k', columns='semilla', values='f1_macro').round(3).to_string())

**Criterio de granularidad.** Se define de forma explícita un F1 macro mínimo aceptable de **0.30** (la mitad de las clases se reconoce con precisión y recall moderados) y una ganancia mínima de **2 veces** sobre el azar 1/k. Se busca el mayor k cuyo mejor modelo cumpla ambos.

In [ ]:
UMBRAL_F1, UMBRAL_GANANCIA = 0.30, 2.0
mejor_por_k = resumen.sort_values('f1_media').groupby('k').tail(1).sort_values('k')
mejor_por_k['cumple'] = (mejor_por_k['f1_media'] >= UMBRAL_F1) & (mejor_por_k['ganancia_vs_azar'] >= UMBRAL_GANANCIA)
print(mejor_por_k[['k', 'modelo', 'f1_media', 'f1_desv', 'acc_media', 'azar', 'ganancia_vs_azar', 'cumple']].to_string(index=False))
k_recomendado = int(mejor_por_k[mejor_por_k.cumple]['k'].max()) if mejor_por_k.cumple.any() else None
print("\nMayor k que cumple el criterio:", k_recomendado)

# Rendimiento por clase del SVM de la Fase 4 (13 clases) para decidir qué agrupar o conservar
print("\nF1 por clase del SVM con 13 clases (Fase 4):")
print(f1_por_clase['SVM'].sort_values(ascending=False).round(3).to_string())

In [ ]:
salida6 = {'semillas': SEMILLAS, 'ks': KS, 'tiempo_total_s': round(tiempo_total_f6, 1),
           'config_rnn': {'max_tokens': 15000, 'dim_emb': 32, 'unidades': 32, 'max_epocas': 4, 'paciencia': 1},
           'criterio': {'f1_macro_minimo': UMBRAL_F1, 'ganancia_minima_vs_azar': UMBRAL_GANANCIA},
           'k_recomendado': k_recomendado,
           'resumen': resumen.to_dict('records'),
           'tabla_larga': tabla_larga.round(5).to_dict('records'),
           'sorteos': sorteos.to_dict('records')}
with open('results/fase6.json', 'w') as archivo:
    json.dump(salida6, archivo, indent=2, default=str)
tabla_larga.to_csv('results/fase6_tabla_larga.csv', index=False)

**Conclusión de negocio (Fase 6).** El F1 macro cae de forma sistemática al aumentar el número de emociones, y el SVM es el mejor modelo en todos los k. Promedio de 3 semillas (42, 43, 44) del SVM: 0.543 con k=3, 0.393 con k=5, 0.316 con k=7, 0.258 con k=9 y 0.229 con k=11 (NB: 0.480, 0.304, 0.237, 0.190, 0.169; RNN pequeña: 0.494, 0.329, 0.259, 0.194, 0.169). La línea de azar 1/k también baja (0.333, 0.200, 0.143, 0.111, 0.091), así que la ganancia relativa sobre el azar del SVM sube de 1.6 veces (k=3) a 2.5 veces (k=11): al tener más clases el problema se vuelve más difícil en términos absolutos, pero el modelo sigue aprendiendo algo; en cifras absolutas, la distancia sobre el azar es mayor con pocas clases (+0.21 con k=3, +0.14 con k=11). La dispersión entre semillas es grande con pocas clases (desviación del SVM de 0.0945 con k=3, 0.009 con k=11) porque depende de cuáles emociones salgan: por ejemplo, con k=3 el SVM da 0.448 (anger, relief, surprise), 0.544 (hate, neutral, worry) y 0.637 (happiness, love, sadness). Accuracy y F1 macro divergen cuando salen clases muy desiguales: NB con la semilla 42 y k=7 (anger=16, boredom=27, empty=121 junto a neutral=1280 y happiness=776) tiene accuracy 0.492 y F1 macro 0.205, porque la clase neutral es 46 % de la prueba y las raras casi no se reconocen. Criterio fijado de antemano: F1 macro mínimo de 0.30 y ganancia de al menos 2 veces sobre el azar; el mayor k que lo cumple (con el mejor modelo, SVM) es 7 (F1 0.316, 2.2 veces el azar); k=5 lo supera en F1 pero queda en 1.96 veces el azar, y k=9 y 11 no llegan a 0.30. **Recomendación:** trabajar con unas 5 a 7 categorías, no con 13. Conservar las emociones grandes y con señal propia (neutral, worry, sadness, happiness, love, hate) y agrupar las pequeñas o solapadas (fun, enthusiasm y relief con happiness; anger y boredom con hate o sadness; empty y surprise en «otras»), ya que en 13 clases anger, boredom, empty y enthusiasm tienen F1 de 0.08 o menos con el SVM. Advertencia: el experimento sortea subconjuntos al azar de las emociones originales, no prueba agrupaciones; la agrupación sugerida es una hipótesis que habría que medir. Tiempo del experimento: 5.8 min con RNN reducida (embedding 32, GRU de 32, máx. 4 épocas, paciencia 1), por lo que la RNN de este experimento es más débil que la de la Fase 5.

<a id="fase-7"></a>
## Fase 7: Big Data: PySpark, MLlib y MapReduce

Repliquen la clasificación con un **Pipeline de Spark MLlib** (tokenización, stopwords, TF-IDF, Naive Bayes o regresión logística) y un **conteo de palabras estilo MapReduce** con RDD. Muestren el plan con `explain()`, una consulta equivalente en **Spark SQL** (`createOrReplaceTempView`) y una **función de ventana** (top-5 palabras por emoción con `row_number()`). Relacionen cada paso con Hadoop (HDFS para almacenamiento, MapReduce frente a Spark en memoria) y describan cómo se desplegaría en un clúster en AWS (EMR, datos en S3). **Spark 4:** expliquen qué cambia con Spark Connect (el notebook como cliente de un clúster remoto, `SparkSession.builder.remote(...)`) y por qué en ese modo no hay RDD.

> **Pregunta de la guía:** Plan de explain(), consulta Spark SQL con función de ventana, conteo MapReduce y despliegue en EMR con S3.

**Qué analizamos:** Se replica la clasificación de emociones con un Pipeline de Spark MLlib (tokenizar, quitar stopwords, TF-IDF y Naive Bayes) sobre la MISMA partición 70/15/15 de sklearn, y se hace un conteo de palabras estilo MapReduce con RDD. Además se lee el plan con explain(), se consulta con Spark SQL y una función de ventana (top-5 palabras por emoción), se comparan tiempos con sklearn y se estima cuándo un clúster (EMR + S3) se justifica de verdad.

**Spark en modo local y la misma partición que sklearn.** Se crea la sesión de Spark (4 GB para el driver) y se pasan a Spark exactamente los mismos tweets de entrenamiento, validación y prueba de la Fase 4 (de pandas a Spark). Así la comparación con sklearn es justa: mismo dato, misma partición, misma semilla. En producción este paso sería `spark.read.csv('s3a://...')` o HDFS, no `createDataFrame` desde pandas.

In [ ]:
import os, sys, io, re, time, json, contextlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.ml import Pipeline as SparkPipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

os.environ.setdefault('PYSPARK_PYTHON', sys.executable)   # los workers de Spark usan el mismo Python del notebook
t0 = time.time()
spark = (SparkSession.builder.appName('emociones').master('local[*]')
         .config('spark.driver.memory', '4g').getOrCreate())  # 4 GB: el valor por defecto (1 GB) se queda corto con TF-IDF
spark.sparkContext.setLogLevel('ERROR')                      # menos ruido en la salida
sdf = spark.createDataFrame(df[['texto_limpio', 'sentiment']])  # corpus completo (para MapReduce, SQL y ventana)

def a_spark(textos, etiquetas):
    """Pasa una partición de sklearn (Series) a un DataFrame de Spark con las mismas columnas del corpus."""
    return spark.createDataFrame(pd.DataFrame({'texto_limpio': textos.values, 'sentiment': etiquetas.values}))

sdf_train, sdf_val, sdf_test = a_spark(X_train, y_train), a_spark(X_val, y_val), a_spark(X_test, y_test)
t_sesion = time.time() - t0
print(f"Spark {spark.version} listo en {t_sesion:.1f} s | núcleos locales: {spark.sparkContext.defaultParallelism}")
print(f"Filas -> corpus: {sdf.count()}, train: {sdf_train.count()}, val: {sdf_val.count()}, test: {sdf_test.count()}")
print("Mismas cifras que sklearn:", (sdf_train.count(), sdf_val.count(), sdf_test.count()) == (len(X_train), len(X_val), len(X_test)))

**Pipeline de MLlib.** Cada etapa transforma una columna del DataFrame y la siguiente la consume: `StringIndexer` (emoción → número, en orden alfabético igual que `ETIQUETAS`), `RegexTokenizer` (texto → palabras en minúscula), `StopWordsRemover` (quita «the», «and»…), `HashingTF` (cuenta palabras en un vector de 2^15 posiciones) e `IDF` (pondera: palabras raras pesan más). El clasificador es `NaiveBayes` multinomial. Se ajusta un solo hiperparámetro (`smoothing`) con la validación y la prueba se usa UNA vez.

In [ ]:
# Etapas de características (comunes a todas las pruebas de smoothing)
indexador = StringIndexer(inputCol='sentiment', outputCol='label', stringOrderType='alphabetAsc')  # orden alfabético = ETIQUETAS
tokenizador = RegexTokenizer(inputCol='texto_limpio', outputCol='palabras', pattern=r"[^a-zA-Z']+", toLowercase=True)
sin_stop = StopWordsRemover(inputCol='palabras', outputCol='sin_stop')   # lista de stopwords en inglés por defecto
tf = HashingTF(inputCol='sin_stop', outputCol='tf', numFeatures=2**15)
idf = IDF(inputCol='tf', outputCol='features')
etapas_car = [indexador, tokenizador, sin_stop, tf, idf]

# Se ajustan las características UNA vez sobre train y se reutilizan para probar varios smoothing en validación
t0 = time.time()
modelo_car = SparkPipeline(stages=etapas_car).fit(sdf_train)
train_f = modelo_car.transform(sdf_train).select('features', 'label').cache()
val_f = modelo_car.transform(sdf_val).select('features', 'label').cache()
t_car = time.time() - t0
print(f"Características listas en {t_car:.1f} s; el orden de clases coincide con ETIQUETAS:",
      modelo_car.stages[0].labels == list(ETIQUETAS))

def f1_macro_spark(pred_sdf):
    """F1 macro calculado con sklearn sobre las predicciones recogidas (MLlib 'f1' es PONDERADO, no macro)."""
    p = pred_sdf.select('label', 'prediction').toPandas()
    return f1_score(p['label'], p['prediction'], average='macro'), accuracy_score(p['label'], p['prediction'])

t0 = time.time()
filas = []
for suavizado in [0.1, 0.3, 1.0]:                      # smoothing de Laplace del Naive Bayes
    nb = NaiveBayes(smoothing=suavizado, modelType='multinomial').fit(train_f)
    f1_v, acc_v = f1_macro_spark(nb.transform(val_f))
    filas.append({'smoothing': suavizado, 'f1_macro_val': round(f1_v, 4), 'accuracy_val': round(acc_v, 4)})
tabla_val = pd.DataFrame(filas)
t_ajuste = time.time() - t0   # tiempo de probar los 3 valores de smoothing
print(tabla_val.to_string(index=False))
mejor_smoothing = float(tabla_val.sort_values('f1_macro_val', ascending=False).iloc[0]['smoothing'])
print("Mejor smoothing en validación:", mejor_smoothing)

In [ ]:
# Modelo final: Pipeline COMPLETO (características + NB) ajustado sobre train; se mide su tiempo de entrenamiento
nb_final = NaiveBayes(smoothing=mejor_smoothing, modelType='multinomial')
t0 = time.time()
pipeline_spark = SparkPipeline(stages=etapas_car + [nb_final]).fit(sdf_train)
tiempo_spark = time.time() - t0

# Evaluación en PRUEBA, una sola vez
t0 = time.time()
pred_spark = pipeline_spark.transform(sdf_test)
f1_spark, acc_spark = f1_macro_spark(pred_spark)
f1_pond_spark = MulticlassClassificationEvaluator(labelCol='label', metricName='f1').evaluate(pred_spark)  # ponderado, solo informativo
t_pred_spark = time.time() - t0
print(f"Spark MLlib (TF-IDF + NB, smoothing={mejor_smoothing}) en prueba:")
print(f"  F1 macro = {f1_spark:.4f} (calculado con sklearn sobre las predicciones)  |  accuracy = {acc_spark:.4f}")
print(f"  F1 ponderado de MLlib = {f1_pond_spark:.4f} (NO es el macro; solo se informa)")
print(f"  Entrenamiento del Pipeline: {tiempo_spark:.1f} s  |  predicción + evaluación: {t_pred_spark:.1f} s")

# Se añade la fila a la tabla global de resultados (fase 7)
nombre_spark = f'Spark MLlib (TF-IDF + NB, smoothing={mejor_smoothing})'
resultados = pd.concat([resultados, pd.DataFrame([{'modelo': nombre_spark, 'fase': 7, 'f1_macro': round(f1_spark, 4),
                                                   'accuracy': round(acc_spark, 4), 'tiempo_s': round(tiempo_spark, 2)}])],
                       ignore_index=True)
print(resultados.to_string(index=False))

**Conteo de palabras estilo MapReduce con RDD.** Fase *map*: cada tweet se convierte en pares (palabra, 1) con `flatMap` + `map`. *Shuffle*: Spark agrupa los pares con la misma palabra en la misma partición (en Hadoop esto se escribe a disco). Fase *reduce*: `reduceByKey` suma los 1 de cada palabra (primero dentro de cada partición, como un *combiner*, y luego entre particiones). Se filtran stopwords y palabras de menos de 3 letras para que el top sea interpretable. Después se repite con DataFrame para comparar.

In [ ]:
parar = set(StopWordsRemover.loadDefaultStopWords('english'))   # mismas stopwords que el pipeline
parar_bc = spark.sparkContext.broadcast(parar)                   # se envía una copia a cada ejecutor (como la caché distribuida de Hadoop)

def palabras_de(texto):
    """Fase MAP (parte 1): un tweet -> lista de palabras útiles."""
    return [p for p in re.findall(r"[a-z']+", texto.lower()) if len(p) > 2 and p not in parar_bc.value]

# --- MapReduce con RDD ---
t0 = time.time()
rdd = sdf.rdd.map(lambda fila: fila['texto_limpio'])             # RDD de textos
conteo_rdd = (rdd.flatMap(palabras_de)                           # MAP: tweet -> palabras
                 .map(lambda w: (w, 1))                          # MAP: palabra -> (palabra, 1)
                 .reduceByKey(lambda a, b: a + b))               # SHUFFLE + REDUCE: suma por palabra
top15_rdd = conteo_rdd.takeOrdered(15, key=lambda kv: (-kv[1], kv[0]))   # top 15 (desempate alfabético)
t_rdd = time.time() - t0

# --- El mismo conteo con DataFrame (Catalyst lo optimiza) ---
t0 = time.time()
palabras_df = (sdf.select(F.explode(F.split(F.lower('texto_limpio'), r"[^a-z']+")).alias('palabra'))
                  .filter((F.length('palabra') > 2) & (~F.col('palabra').isin(list(parar)))))
top15_df = palabras_df.groupBy('palabra').count().orderBy(F.desc('count'), 'palabra').limit(15).collect()
t_df = time.time() - t0

print(f"RDD (MapReduce): {t_rdd:.1f} s | DataFrame: {t_df:.1f} s")
print("Top 15 con RDD:", top15_rdd)
print("Mismo top 15 con DataFrame:", [(r['palabra'], r['count']) for r in top15_df] == top15_rdd)
print(f"Palabras distintas: {conteo_rdd.count()} | particiones del RDD: {rdd.getNumPartitions()}")

**Qué se analiza:** las 15 palabras más frecuentes del corpus (sin stopwords), obtenidas con el conteo MapReduce. Sirve para ver que el resultado del RDD es el vocabulario dominante y para comprobar visualmente que el filtro dejó palabras con significado.

In [ ]:
palabras, cuentas = zip(*top15_rdd[::-1])        # invertido para que la más frecuente quede arriba
fig, ax = plt.subplots(figsize=(7, 5))
colores = [ACENTO if i == len(palabras) - 1 else GRIS for i in range(len(palabras))]   # ACENTO solo para la más frecuente
ax.barh(palabras, cuentas, color=colores)
for i, v in enumerate(cuentas):
    ax.text(v, i, f" {v:,}", va='center', fontsize=8, color=TINTA)
ax.set_xlabel('Apariciones en el corpus', color=TINTA)
ax.set_title('Top 15 palabras (conteo MapReduce con RDD)', color=TINTA, loc='left')
ax.spines[['top', 'right']].set_visible(False); ax.grid(False)
plt.tight_layout(); plt.savefig('figuras/fase7_top_palabras_mapreduce.png', dpi=150); plt.show()

**Plan de ejecución con `explain()`.** Cada consulta pasa por cuatro planes: *Parsed* (lo que se escribió), *Analyzed* (con columnas y tipos resueltos), *Optimized* (tras las reglas de Catalyst: poda de columnas, filtros adelantados) y *Physical* (los operadores reales). Se busca `Exchange`: es el **shuffle**, el equivalente a la fase shuffle de MapReduce, donde los datos viajan entre particiones y se paga en red/disco.

In [ ]:
def texto_plan(consulta, modo):
    """Captura lo que imprime explain() para poder buscar operadores dentro del plan."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        consulta.explain(mode=modo)
    return buf.getvalue()

agregacion = sdf.groupBy('sentiment').agg(F.count('*').alias('n_tweets'), F.avg(F.length('texto_limpio')).alias('long_media'))
plan_ext = texto_plan(agregacion, 'extended')
print(plan_ext)
plan_fmt = texto_plan(agregacion, 'formatted')
print(plan_fmt)
def n_shuffles(plan_texto):
    """Cuenta los operadores Exchange del plan físico (en 'extended' solo aparecen en el árbol, una vez cada uno)."""
    return len(re.findall(r'Exchange (?:hash|range)?[Pp]artitioning|Exchange SinglePartition', plan_texto.split('== Physical Plan ==')[-1]))
n_exchange = n_shuffles(plan_ext)
print(f"Operadores Exchange (shuffle) en el plan físico: {n_exchange}")
print("HashAggregate presente 2 veces (parcial antes del shuffle y final después):", plan_fmt.count('HashAggregate') >= 2)

**Cómo leer el plan anterior.** (1) *Parsed/Analyzed*: `Aggregate [sentiment], [sentiment, count(1), avg(length(texto_limpio))]` sobre la relación local creada desde pandas. (2) *Optimized*: aquí es idéntico al Analyzed porque la consulta ya es mínima; con filtros o `select` Catalyst los empuja hacia la lectura y poda columnas (con Parquet se vería en `PushedFilters`/`ReadSchema`). (3) *Physical*: `HashAggregate (partial)` → `Exchange hashpartitioning(sentiment, ...)` → `HashAggregate (final)`. Ese `Exchange` es el shuffle: primero cada partición agrega lo suyo (como el *combiner* de MapReduce) y solo viajan los totales parciales por emoción. Lo mismo que en el conteo con `reduceByKey`, pero con el plan elegido por el optimizador.

**Spark SQL.** La misma agregación escrita en SQL: se registra el DataFrame como vista temporal con `createOrReplaceTempView` y se consulta con `spark.sql`. SQL y DataFrame generan el mismo plan lógico, así que el rendimiento es igual; cambia solo la comodidad (SQL para analistas, DataFrame para código).

In [ ]:
sdf.createOrReplaceTempView('tweets')
consulta_sql = """
    SELECT sentiment, COUNT(*) AS n_tweets, ROUND(AVG(LENGTH(texto_limpio)), 1) AS long_media
    FROM tweets
    GROUP BY sentiment
    ORDER BY n_tweets DESC
"""
res_sql = spark.sql(consulta_sql).toPandas()
print(res_sql.to_string(index=False))

# Verificación contra pandas (debe dar lo mismo)
ref = df.groupby('sentiment')['texto_limpio'].agg(n_tweets='size', long_media=lambda s: s.str.len().mean()).round(1)
comp = res_sql.set_index('sentiment').join(ref, rsuffix='_pandas')
print("Spark SQL = pandas (conteos):", (comp['n_tweets'] == comp['n_tweets_pandas']).all(),
      "| longitudes (tolerancia 0.1):", np.allclose(comp['long_media'], comp['long_media_pandas'], atol=0.1))
plan_sql = texto_plan(spark.sql(consulta_sql), 'extended')
print("Exchange en el plan de la consulta SQL:", n_shuffles(plan_sql), "(uno para agrupar por emoción y otro para ordenar globalmente por n_tweets)")

**Función de ventana: top-5 palabras por emoción.** Primero se cuentan palabras por emoción (`explode(split(...))` + `groupBy`) y luego `row_number()` numera las palabras dentro de cada emoción (`Window.partitionBy(sentiment).orderBy(desc(conteo))`); se conservan las posiciones 1 a 5. Se filtran stopwords y palabras de menos de 3 letras para que la lista sea interpretable. La misma consulta se escribe también en SQL para mostrar que es equivalente.

In [ ]:
conteo_emo = (sdf.select('sentiment', F.explode(F.split(F.lower('texto_limpio'), r"[^a-z']+")).alias('palabra'))
                                   .filter((F.length('palabra') > 2) & (~F.col('palabra').isin(list(parar))))
                                   .groupBy('sentiment', 'palabra').count())
ventana = Window.partitionBy('sentiment').orderBy(F.desc('count'), F.asc('palabra'))   # desempate alfabético = resultado repetible
top5 = conteo_emo.withColumn('posicion', F.row_number().over(ventana)).filter(F.col('posicion') <= 5)
top5_pd = top5.orderBy('sentiment', 'posicion').toPandas()

# Una fila por emoción: "palabra (conteo), ..."
resumen_top5 = (top5_pd.assign(par=lambda d: d['palabra'] + ' (' + d['count'].astype(str) + ')')
                       .groupby('sentiment')['par'].apply(lambda s: ', '.join(s)))
for emo, fila in resumen_top5.items():
    print(f"{emo:<11}{fila}")

# Versión SQL equivalente
conteo_emo.createOrReplaceTempView('conteo_emocion')
top5_sql = spark.sql("""
    SELECT sentiment, palabra, count, posicion FROM (
        SELECT sentiment, palabra, count,
               ROW_NUMBER() OVER (PARTITION BY sentiment ORDER BY count DESC, palabra ASC) AS posicion
        FROM conteo_emocion) t
    WHERE posicion <= 5
    ORDER BY sentiment, posicion""").toPandas()
print("SQL con ROW_NUMBER() = versión DataFrame:", top5_sql.equals(top5_pd[['sentiment', 'palabra', 'count', 'posicion']].astype(top5_sql.dtypes.to_dict())))
plan_ventana = texto_plan(top5, 'extended')
print("Operador Window en el plan:", 'Window' in plan_ventana, "| Exchange:", n_shuffles(plan_ventana),
      "(shuffle por sentiment para reunir cada emoción en una partición)")

**Qué se analiza:** cuánto tardan sklearn y Spark en entrenar sobre el MISMO dato (unos 40 000 tweets; ≈ 2.6 MB de texto limpio, el archivo CSV original pesa ≈ 3.8 MB) y qué F1 macro obtiene cada uno. Se espera que Spark local sea MÁS lento: el costo fijo de la JVM, la planificación de tareas y la serialización no se amortizan con un conjunto tan pequeño.

In [ ]:
# Comparación del MISMO algoritmo (TF-IDF + Naive Bayes, alpha=1) en sklearn, para que sea justa con Spark
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
t0 = time.time()
nb_sk = Pipeline([('tfidf', TfidfVectorizer(stop_words='english')), ('nb', MultinomialNB(alpha=1.0))]).fit(X_train, y_train)
tiempo_sk_nb = time.time() - t0
t0 = time.time()
pred_sk_nb = nb_sk.predict(X_test)
t_pred_sk_nb = time.time() - t0
f1_sk_nb = f1_score(y_test, pred_sk_nb, average='macro')
acc_sk_nb = accuracy_score(y_test, pred_sk_nb)
fila_nb = pd.DataFrame([{'modelo': 'sklearn TF-IDF + NB (mismo algoritmo)', 'fase': 4, 'f1_macro': round(f1_sk_nb, 4),
                         'accuracy': round(acc_sk_nb, 4), 'tiempo_s': round(tiempo_sk_nb, 2)}])
sk = pd.concat([resultados[resultados['fase'].isin([4, 7])], fila_nb], ignore_index=True)   # modelos clásicos de sklearn y Spark
sk['motor'] = np.where(sk['fase'] == 7, 'Spark', 'sklearn')
sk = sk.drop_duplicates('modelo', keep='first').reset_index(drop=True)

# Tiempo de punta a punta de Spark (sesión + características + ajuste de smoothing + entrenamiento final + predicción)
t_total_spark = t_sesion + t_car + t_ajuste + tiempo_spark + t_pred_spark
print(f"sklearn NB: entrenar {tiempo_sk_nb:.2f} s + predecir {t_pred_sk_nb:.2f} s | Spark: entrenar el Pipeline {tiempo_spark:.1f} s, "
      f"y de punta a punta (sesión, características, ajuste, entrenamiento, predicción) {t_total_spark:.1f} s")
print(f"Spark local tarda {t_total_spark / (tiempo_sk_nb + t_pred_sk_nb):.0f} veces más que sklearn NB en el flujo completo")
print(sk[['modelo', 'motor', 'f1_macro', 'accuracy', 'tiempo_s']].to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 3.8))
nombres = [m if len(m) < 34 else m[:31] + '...' for m in sk['modelo']]
colores = [ACENTO if mo == 'Spark' else GRIS for mo in sk['motor']]   # ACENTO = Spark
ax.barh(nombres, sk['tiempo_s'], color=colores)
for i, (t, f) in enumerate(zip(sk['tiempo_s'], sk['f1_macro'])):
    ax.text(t, i, f"  {t:.1f} s | F1 macro {f:.3f}", va='center', fontsize=8, color=TINTA)
ax.set_xlim(0, sk['tiempo_s'].max() * 1.6)
ax.set_xlabel('Tiempo de entrenamiento (s)', color=TINTA)
ax.set_title('Mismo dato, mismo split: sklearn vs Spark MLlib (local)', color=TINTA, loc='left')
ax.spines[['top', 'right']].set_visible(False); ax.grid(False)
plt.tight_layout(); plt.savefig('figuras/fase7_tiempos_sklearn_vs_spark.png', dpi=150); plt.show()

**Cuándo basta un solo nodo y cuándo se justifica Spark.** Se mide el tamaño real por tweet y cuánta memoria ocupa la matriz TF-IDF en sklearn respecto al texto original; con ese factor se estiman tres escenarios (10 mil, 1 millón y 100 millones de tweets al día) para un año de historia. Los supuestos van explícitos en el código: el registro completo (texto + usuario + fecha + metadatos) se asume de 500 bytes y la máquina de referencia es un servidor de 64 GB de RAM donde se acepta usar hasta la mitad para el modelo.

In [ ]:
# --- Medidas reales del corpus ---
bytes_texto = df['texto_limpio'].str.encode('utf-8').str.len()
bytes_medio = float(bytes_texto.mean())
mb_corpus = float(bytes_texto.sum()) / 1e6
mb_crudo = float(df['content'].str.encode('utf-8').str.len().sum()) / 1e6   # texto original sin limpiar
tfidf_ref = TfidfVectorizer(min_df=2, sublinear_tf=True)
M = tfidf_ref.fit_transform(X_train)
bytes_matriz = M.data.nbytes + M.indices.nbytes + M.indptr.nbytes
factor_ram = bytes_matriz / X_train.str.encode('utf-8').str.len().sum()   # RAM de la matriz TF-IDF por byte de texto
print(f"Texto por tweet: {bytes_medio:.0f} bytes en promedio | corpus completo: {mb_corpus:.2f} MB de texto limpio ({mb_crudo:.2f} MB de texto original)")
print(f"Matriz TF-IDF (train): {bytes_matriz / 1e6:.1f} MB = {factor_ram:.1f}x el tamaño del texto original")

# --- Supuestos explícitos (NO medidos) ---
BYTES_REGISTRO = 500          # supuesto: texto + usuario + fecha + metadatos de un tweet (bytes)
DIAS_HISTORIA = 365           # supuesto: se quiere reentrenar con un año de historia
RAM_NODO_GB = 64              # supuesto: un servidor de 64 GB de RAM
USO_MAX_RAM = 0.5             # supuesto: se acepta usar hasta el 50 % de la RAM para datos + matriz
escenarios = []
for nombre, tweets_dia in [('10 mil/día', 10_000), ('1 millón/día', 1_000_000), ('100 millones/día', 100_000_000)]:
    gb_dia = tweets_dia * BYTES_REGISTRO / 1e9
    gb_anio = gb_dia * DIAS_HISTORIA
    gb_ram = gb_anio * (1 + factor_ram)                    # datos crudos + matriz TF-IDF en memoria
    cabe = gb_ram <= RAM_NODO_GB * USO_MAX_RAM
    gb_ram_30d = gb_dia * 30 * (1 + factor_ram)            # alternativa: reentrenar solo con una ventana de 30 días
    escenarios.append({'escenario': nombre, 'tweets_dia': tweets_dia, 'GB_dia': round(gb_dia, 3), 'GB_anio': round(gb_anio, 1),
                       'RAM_estimada_GB': round(gb_ram, 1), 'cabe_en_un_nodo': bool(cabe),
                       'RAM_30d_GB': round(gb_ram_30d, 1), 'cabe_30d': bool(gb_ram_30d <= RAM_NODO_GB * USO_MAX_RAM)})
tabla_esc = pd.DataFrame(escenarios)
print(tabla_esc.to_string(index=False))
print(f"Referencia: el corpus del ejercicio pesa {mb_corpus:.1f} MB de texto; "
      f"un año a 10 mil/día son {tabla_esc.loc[0, 'GB_anio']} GB.")

**Qué se analiza:** cuánta memoria necesitaría cada escenario (un año de historia) frente al límite que se acepta usar en un nodo de 64 GB. Donde la barra supera la línea punteada, un solo nodo ya no alcanza y se justifica distribuir con Spark.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
colores = [GRIS if c else ACENTO for c in tabla_esc['cabe_en_un_nodo']]   # ACENTO = ya no cabe en un nodo
ax.bar(tabla_esc['escenario'], tabla_esc['RAM_estimada_GB'], color=colores)
ax.set_yscale('log')
ax.axhline(RAM_NODO_GB * USO_MAX_RAM, color=TINTA, linestyle='--', linewidth=1)
ax.text(2.45, RAM_NODO_GB * USO_MAX_RAM * 1.15, f"límite: {RAM_NODO_GB * USO_MAX_RAM:.0f} GB (50 % de 64 GB)", ha='right', fontsize=8, color=TINTA)
for i, v in enumerate(tabla_esc['RAM_estimada_GB']):
    ax.text(i, v * 1.12, f"{v:,.1f} GB", ha='center', fontsize=8, color=TINTA)
ax.set_ylabel('RAM estimada (GB, escala log)', color=TINTA)
ax.set_title('Un año de historia: ¿cabe en un solo nodo?', color=TINTA, loc='left')
ax.spines[['top', 'right']].set_visible(False); ax.grid(False)
plt.tight_layout(); plt.savefig('figuras/fase7_volumen_un_nodo_vs_cluster.png', dpi=150); plt.show()

**De Hadoop a Spark, paso por paso.**
- **Almacenamiento: HDFS ↔ S3.** HDFS parte cada archivo en bloques de 128 MB, los replica 3 veces y los reparte en DataNodes; el NameNode guarda el mapa de bloques. En AWS se usa S3 (objetos, `s3a://bucket/ruta`): almacenamiento separado del cómputo, más barato, durable y sin NameNode que administrar. Por eso `spark.createDataFrame(df)` de este notebook sería `spark.read.csv('s3a://bucket/tweets/')` o, mejor, `spark.read.parquet(...)`.
- **Cómputo: MapReduce ↔ Spark.** MapReduce escribe a disco el resultado de cada fase (map → disco → shuffle → reduce → disco): robusto, pero lento en trabajos iterativos. Spark construye un DAG de transformaciones perezosas y mantiene datos en memoria entre etapas (`cache()`), por eso un Pipeline de MLlib con varias pasadas sobre los datos es mucho más rápido en clúster. El `reduceByKey` de arriba es el mismo patrón map/shuffle/reduce.
- **Recursos: YARN ↔ EMR.** YARN reparte CPU y memoria entre aplicaciones del clúster. En AWS, **EMR** lo levanta ya configurado: un nodo maestro (coordina y corre el driver), nodos *core* (cómputo + almacenamiento HDFS temporal) y nodos *task* (solo cómputo, ideales para instancias *spot*, que cuestan bastante menos pero pueden interrumpirse). **EMR Serverless** evita administrar nodos: se envía el trabajo y AWS ajusta los recursos.
- **Despliegue típico de este trabajo:** (1) subir `tweet_emotions.csv` (o mejor Parquet) a S3; (2) empaquetar el script con el Pipeline; (3) `spark-submit --deploy-mode cluster --master yarn s3://bucket/codigo/entrenar.py` en un paso (*step*) de EMR; (4) guardar el modelo y las predicciones en S3 en Parquet; (5) permisos con roles IAM (el rol de EMR lee/escribe solo ese bucket); (6) apagar el clúster al terminar (clústeres transitorios) y usar spot en nodos task para bajar el costo. Para datos personales, además, cifrado en S3 y región/retención acordes con la LOPDP.

**Spark 4 y Spark Connect.** Hasta Spark 3 el notebook (el *driver*) corría dentro de la misma JVM que Spark: el cliente tenía acceso directo a `SparkContext` y a los RDD. Con **Spark Connect** (nuevo en Spark 3.4 y modo recomendado en Spark 4) el notebook pasa a ser un **cliente ligero** de un servidor Spark remoto: se conecta por gRPC con `SparkSession.builder.remote('sc://host:15002')`, y lo único que viaja son **planes lógicos de DataFrame/SQL** (en formato Protocol Buffers); el servidor los optimiza y ejecuta en el clúster y devuelve los resultados (en formato Arrow). Ventajas: el cliente es liviano (no necesita Java), varios usuarios comparten un clúster, se actualiza el servidor sin romper los notebooks y un fallo del cliente no tumba el driver.

**Por qué en ese modo no hay RDD ni `SparkContext`:** un RDD se define con funciones Python arbitrarias (`lambda`) aplicadas sobre particiones y con acceso directo a la JVM, y el protocolo solo transporta planes de DataFrame/SQL, no funciones arbitrarias sobre particiones. Por eso, en Connect, `sdf.rdd`, `spark.sparkContext` y `sc.parallelize` no existen y el conteo MapReduce de arriba se escribe con `explode/split/groupBy` (que sí viaja como plan) o con UDF/`mapInPandas`. MLlib sí funciona en Connect para los algoritmos soportados.

In [ ]:
# --- Spark Connect (Spark 4): el notebook como CLIENTE de un clúster remoto. NO se ejecuta aquí. ---
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.remote('sc://mi-servidor-spark:15002').getOrCreate()  # gRPC; solo viajan planes DataFrame/SQL
# sdf = spark.read.parquet('s3a://mi-bucket/tweets/')       # el servidor (EMR) lee de S3, el notebook solo recibe el resultado
# sdf.groupBy('sentiment').count().show()                    # funciona: es un plan de DataFrame
# sdf.rdd.flatMap(...)                                       # ERROR en Connect: no hay RDD ni SparkContext en el cliente
# # Equivalente local sin clúster: SparkSession.builder.config('spark.api.mode', 'connect').master('local[*]').getOrCreate()
print("Spark Connect: snippet comentado (no ejecutado). Sesión clásica 'spark' sigue activa para la Fase 9:", spark.version)

In [ ]:
# Cifras clave a JSON (results/fase7.json)
resumen7 = {
    'spark_version': spark.version,
    'filas': {'corpus': int(len(df)), 'train': int(len(X_train)), 'val': int(len(X_val)), 'test': int(len(X_test))},
    'validacion_smoothing': tabla_val.to_dict('records'),
    'mejor_smoothing': mejor_smoothing,
    'mllib_test': {'f1_macro_sklearn': round(float(f1_spark), 4), 'accuracy': round(float(acc_spark), 4),
                   'f1_ponderado_mllib': round(float(f1_pond_spark), 4),
                   'tiempo_entrenamiento_s': round(tiempo_spark, 2), 'tiempo_prediccion_eval_s': round(t_pred_spark, 2)},
    'tiempo_sesion_spark_s': round(t_sesion, 1),
    'tiempos_spark_s': {'caracteristicas': round(t_car, 1), 'ajuste_smoothing': round(t_ajuste, 1), 'entrenamiento_final': round(tiempo_spark, 2),
                        'prediccion_eval': round(t_pred_spark, 2), 'total_punta_a_punta': round(t_total_spark, 1)},
    'sklearn_nb_mismo_algoritmo': {'f1_macro': round(float(f1_sk_nb), 4), 'accuracy': round(float(acc_sk_nb), 4),
                                   'entrenar_s': round(tiempo_sk_nb, 2), 'predecir_s': round(t_pred_sk_nb, 2)},
    'comparacion_tiempos': sk[['modelo', 'motor', 'f1_macro', 'accuracy', 'tiempo_s']].to_dict('records'),
    'mapreduce': {'tiempo_rdd_s': round(t_rdd, 2), 'tiempo_dataframe_s': round(t_df, 2),
                  'top15': [[w, int(c)] for w, c in top15_rdd], 'palabras_distintas': int(conteo_rdd.count())},
    'plan_explain': {'exchange_agregacion': int(n_exchange), 'exchange_sql': int(n_shuffles(plan_sql)),
                     'exchange_ventana': int(n_shuffles(plan_ventana))},
    'sql_por_emocion': res_sql.to_dict('records'),
    'top5_por_emocion': resumen_top5.to_dict(),
    'volumen': {'bytes_texto_por_tweet': round(bytes_medio, 1), 'mb_corpus_texto': round(mb_corpus, 2), 'mb_texto_original': round(mb_crudo, 2),
                'factor_ram_tfidf': round(float(factor_ram), 2), 'supuestos': {'bytes_registro': BYTES_REGISTRO, 'dias_historia': DIAS_HISTORIA,
                'ram_nodo_gb': RAM_NODO_GB, 'uso_max_ram': USO_MAX_RAM}, 'escenarios': tabla_esc.to_dict('records')},
}
with open('results/fase7.json', 'w', encoding='utf-8') as f:
    json.dump(resumen7, f, ensure_ascii=False, indent=2, default=str)
print("Guardado results/fase7.json")

**Conclusión de negocio (Fase 7).**
1. **MLlib replica la clasificación, pero no la mejora.** Con la misma partición que sklearn (27 781 train / 5 953 val / 5 954 prueba), el Pipeline de Spark (tokenizar, stopwords, HashingTF 2^15, IDF, Naive Bayes con smoothing = 1.0 elegido en validación) obtiene en prueba F1 macro = 0.142 y accuracy = 0.236 (el F1 ponderado que reporta MLlib, 0.236, no es el macro: aquí se usa el macro calculado con sklearn sobre las predicciones). El mismo algoritmo en sklearn (TF-IDF + NB sin ajustar) da F1 macro = 0.0965 y accuracy = 0.292; la mejor comparación contra el mejor modelo de la Fase 4 está en la tabla `resultados`. La dificultad (13 clases solapadas) la fija el dato, no el motor.
2. **Spark local es MÁS lento para este volumen, y es lo esperable.** En esta corrida, sklearn entrenó el NB en 0.36 s y predijo en 0.09 s; Spark tardó 1.5 s solo en entrenar el Pipeline final y 17.0 s de punta a punta (arrancar la JVM ≈ 7.3 s, características ≈ 3.6 s, probar 3 valores de smoothing ≈ 3.7 s, entrenamiento final y predicción), unas 38 veces más. Los tiempos varían entre corridas con la carga de la máquina, pero el orden de magnitud no. Con ≈ 2.6 MB de texto no se amortiza el costo fijo de Spark (JVM, planificación, serialización).
3. **MapReduce y planes.** El conteo con RDD (flatMap → map → reduceByKey) y el de DataFrame dan exactamente el mismo top 15 (day 3 075, good 2 290, get 2 008, like 1 939, today 1 647…) sobre 29 425 palabras distintas; el RDD tardó ≈ 0.6 s y el DataFrame ≈ 2.2 s (el DataFrame paga la planificación de Catalyst y el explode en la JVM; en un volumen mayor Catalyst suele ganar). `explain()` muestra HashAggregate parcial → `Exchange hashpartitioning(sentiment)` → HashAggregate final: un solo shuffle para agrupar; la consulta SQL (tweets y longitud media por emoción) tiene 2 (agrupar y ordenar) y la ventana top-5 también 2 (agrupar y repartir por emoción para `row_number()`). Los cuellos de botella de un clúster son justo esos Exchange.
4. **Hallazgo de negocio con la ventana.** Las palabras dominantes separan emociones donde hay señal léxica clara (hate: «hate» 210, «sucks»; sadness: «sad» 385, «miss» 340; fun: «lol» 192, «fun» 142; love: «love» 861), pero en neutral, worry, relief o surprise dominan palabras genéricas («day», «get», «good», «like»). Eso anticipa el techo de rendimiento de los modelos bag-of-words.
5. **Recomendación con números.** El texto limpio mide ≈ 66 bytes por tweet; con un registro completo supuesto de 500 bytes y un factor medido de 2.0x para la matriz TF-IDF, un año de historia ocupa: 10 mil tweets/día = 1.8 GB (≈ 5.5 GB de RAM: **un solo nodo basta**, Spark no se justifica); 1 millón/día = 182.5 GB (≈ 550 GB de RAM: **no cabe en un nodo de 64 GB**; incluso con solo 30 días de historia, ≈ 45 GB, queda por encima del 50 % de RAM aceptado, por lo que ahí sí se justifica Spark en EMR con datos en S3 en Parquet); 100 millones/día = 18 250 GB al año (≈ 55 000 GB de RAM): Spark/EMR es obligatorio, además con almacenamiento en S3 y nodos task spot para abaratar. Regla práctica: Spark vale la pena cuando el dato supera cientos de GB o la ventana histórica no cabe en la RAM de una máquina; para este ejercicio (2.6 MB) basta sklearn. Los supuestos (500 bytes por registro, 64 GB de RAM, 50 % de uso, 365 días) están declarados en el código y son estimaciones, no mediciones; en cambio, los 66 bytes por tweet y el factor 2.0x sí salen de los datos.
6. **Spark Connect (Spark 4):** el notebook pasaría a ser un cliente que envía planes de DataFrame/SQL por gRPC a un clúster remoto; por eso el conteo con RDD de esta fase no correría en ese modo y habría que escribirlo con DataFrame (como la versión de la ventana). El snippet quedó comentado y no se ejecutó.

<a id="fase-8"></a>
## Fase 8: Desafío y evaluación de alternativas

Respondan el desafío con evidencia. Completen una tabla de **evaluación de alternativas**: técnica (spaCy, LDA, NB, SVM, RNN, scikit-learn vs. Spark) · ventajas · limitaciones · cuándo la usarían en un escenario profesional o de investigación. Citen lecturas del Tema 3 y documentación oficial.

> **Pregunta de la guía:** Tabla de alternativas y respuesta al desafío (granularidad, modelo y decisión sobre Spark).

**Qué analizamos:** Se responde el desafío con la evidencia de las fases 1 a 7 y 9 y se arma la tabla de evaluación de alternativas (spaCy, LDA, NB, SVM, RNN y scikit-learn frente a Spark), con ventajas, limitaciones, aplicabilidad y fuente.

**Qué se analiza:** primero se juntan en una sola tabla las cifras que sostienen las decisiones (modelos, tiempos y experimento de clases). Todo sale de las variables y archivos de las fases anteriores, no se escribe a mano.

In [ ]:
import json
# Cifras del experimento de clases (fase 6): promedio de 3 semillas por modelo y número de clases
f6 = json.load(open("results/fase6.json"))
tabla_k = pd.DataFrame(f6["resumen"])
print("Modelos con la partición de la fase 4 (13 clases, prueba):")
display(resultados.round(4))
print("\nExperimento de clases (fase 6), resumen:")
display(tabla_k.round(3))
print("\nCriterio de granularidad fijado en la fase 6:", f6["criterio"], "| k recomendado:", f6["k_recomendado"])

**Qué se analiza:** la tabla de evaluación de alternativas pedida por la guía. Cada fila indica ventajas, limitaciones y en qué escenario se usaría la técnica.

In [ ]:
# Tabla de alternativas (las fuentes son lecturas del Tema 3 y documentación oficial)
alternativas = pd.DataFrame([
    ["spaCy (POS, NER, dependencias)",
     "Pipeline listo, rápido con nlp.pipe; da lemas, categorías gramaticales y entidades legibles.",
     "En tweets el NER falla con jerga y emoticones; las proporciones POS casi no separan emociones (fase 2).",
     "Extraer entidades y limpiar texto en un pipeline profesional; no como único insumo para detectar emoción.",
     "Doc. spaCy: spacy.io/usage/linguistic-features; Tema 3, L1 Minería de textos"],
    ["LDA (scikit-learn)",
     "No supervisado, barato e interpretable con palabras clave; sirve para explorar el corpus.",
     "Hay que fijar K; con textos cortos la perplejidad de validación no se estabiliza con K; los temas son mezclas poco nítidas.",
     "Exploración inicial de un corpus largo y sin etiquetas (reclamos, informes).",
     "Doc. scikit-learn (LatentDirichletAllocation); Tema 3, L1"],
    ["Naive Bayes multinomial",
     "Muy rápido (menos de 1 s), simple y buena línea base; accuracy más alta en 13 clases.",
     "Supone independencia entre palabras y favorece a las clases grandes; F1 macro bajo (fase 4).",
     "Prototipo y línea base cuando se necesita una respuesta casi inmediata.",
     "Doc. scikit-learn (naive_bayes); Tema 3, L1"],
    ["SVM lineal (TF-IDF)",
     "Mejor F1 macro de NB y SVM en la fase 4 y en todos los k de la fase 6; entrena en segundos; barato de operar.",
     "No entiende el orden ni el contexto; sigue fallando en las clases raras (anger, boredom).",
     "Opción por defecto para clasificar texto con muchos datos etiquetados y poco presupuesto.",
     "Doc. scikit-learn (svm, LinearSVC); Tema 3, L1"],
    ["RNN (BiGRU en Keras)",
     "Captura el orden de las palabras y puede mejorar con más datos y embeddings preentrenados.",
     "Unas 36 veces más lenta que el SVM, con más parámetros, y no lo superó en este dato (fase 5).",
     "Cuando hay mucho texto etiquetado, GPU y el orden de las palabras importa de verdad.",
     "Doc. Keras (GRU, TextVectorization); Tema 3, L1"],
    ["scikit-learn (un solo nodo)",
     "API simple; todo el flujo cabe en memoria y corre en segundos para 40 000 tweets.",
     "Limitado por la RAM de la máquina; no escala solo a cientos de GB.",
     "Volúmenes de hasta unos pocos GB (nuestro caso hoy).",
     "Doc. scikit-learn; Tema 3, L2 Big Data"],
    ["Spark MLlib (PySpark)",
     "Escala a muchos nodos, lee de HDFS/S3 y guarda en Parquet; el mismo código sirve de 1 a N máquinas.",
     "En un solo equipo fue mucho más lento que sklearn por la sobrecarga de la JVM (fase 7); más difícil de operar.",
     "Cuando el histórico no cabe en una máquina (cientos de GB) o ya existe un clúster EMR/S3.",
     "Doc. Spark (ml-pipeline, spark-connect-overview); Tema 3, L2 Big Data"],
    ["BERTopic / embeddings / LLM (fase 9)",
     "Entienden el significado y no dependen de que las palabras coincidan.",
     "Más costo y latencia, y enviar texto a un servicio externo implica riesgo con la LOPDP; en este entorno no se pudo ejecutar.",
     "Si se justifica por calidad, con datos que puedan salir de la empresa o con modelos locales.",
     "Grootendorst (2022) BERTopic; lectura de la semana ¿LLM o embeddings?; doc. spacy-llm"],
], columns=["tecnica", "ventajas", "limitaciones", "aplicabilidad", "fuente"])
pd.set_option("display.max_colwidth", None)
display(alternativas)
alternativas.to_csv("results/fase8_alternativas.csv", index=False)

**Respuesta al desafío (con la evidencia de este notebook).**

1. **Granularidad:** no recomendamos usar las 13 emociones con confianza. Con 13 clases el mejor F1 macro en prueba fue 0,198 (SVM; azar = 0,077). En el experimento de la fase 6 el SVM promedió 0,543 con 3 clases, 0,393 con 5, 0,316 con 7, 0,258 con 9 y 0,229 con 11. Con el criterio fijado de antemano (F1 macro de al menos 0,30 y al menos el doble del azar), el máximo es 7 clases. Recomendamos 5 a 7 categorías (por ejemplo neutral, worry, sadness, happiness, love y hate, y agrupar el resto). El experimento sortea clases y no prueba agrupaciones, así que esta es una hipótesis que hay que validar.
2. **Modelo:** SVM lineal con TF-IDF. Tuvo el mejor F1 macro (0,198 frente a 0,144 de NB y 0,149 de la RNN) y entrena en unos 2 segundos (1,7 s), frente a unos 62 s de la RNN. NB es el más rápido (menos de 1 s) y tiene la accuracy más alta, pero no detecta bien las clases pequeñas. La RNN no justifica su costo con este dato.
3. **Spark:** hoy no se justifica. Con 40 000 tweets (3,8 MB en CSV) Spark local fue mucho más lento que scikit-learn por la sobrecarga de arranque (17,0 s frente a 0,45 s del flujo completo del NB en sklearn, unas 38 veces). En la estimación de volumen de la fase 7, un millón de tweets al día suma unos 182,5 GB al año (con el supuesto de 500 bytes por registro) y sí superaría una sola máquina al conservar el histórico; con 10 mil al día basta un nodo. Recomendamos empezar con scikit-learn y migrar a Spark sobre EMR con datos en S3 (Parquet) cuando el histórico ya no quepa en memoria.
4. **Limitación seria del dataset:** las etiquetas son ruidosas y muy desbalanceadas (anger tiene 110 tweets y neutral 8 638; la razón es 78,5), la clase `empty` mezcla tweets con emoción evidente y los textos son tweets en inglés de una época concreta. No representan los mensajes en español de clientes en Ecuador. Además, el PSI de 0,35 de la fase 9 muestra que el modelo ya pierde estabilidad entre tramos de tiempo.
5. **Implicación ética:** analizar la emoción de personas es un tratamiento de datos personales (LOPDP). No debe usarse para decisiones automáticas con efecto sobre el cliente sin una persona que revise, y enviar mensajes a una API externa exige base legal y minimización. Con un F1 macro cercano a 0,2, el modelo se equivocaría en la mayoría de las clases raras, por lo que debe servir solo para priorizar la atención y no para etiquetar a las personas.

La **fase de tendencias** usa un modelo de lenguaje. Esta celda lo prepara (Ollama en local, Hugging Face en Colab):

In [ ]:
# ===== Modelo de lenguaje (LLM): Ollama en local o un modelo pequeño de Hugging Face en Colab =====
import json, os, urllib.request                       # json: armar/leer mensajes; urllib: peticiones HTTP a la API de Ollama

def _hosts_ollama():   # lista de direcciones donde buscar Ollama
    """Lugares donde puede estar Ollama. Ej.: en Windows es localhost; desde WSL es la IP del Windows anfitrión."""
    hosts = [os.environ.get("OLLAMA_HOST", ""), "http://localhost:11434"]   # OLLAMA_HOST permite forzar otro equipo
    try:                                              # en WSL, el 'nameserver' de resolv.conf es el Windows anfitrión
        ip = [l.split()[1] for l in open("/etc/resolv.conf") if l.startswith("nameserver")][0]   # IP del Windows anfitrión vista desde WSL
        hosts.append(f"http://{ip}:11434")            # ej.: http://172.29.80.1:11434
    except Exception:                                 # en Colab o Windows no hay WSL: se ignora
        pass   # sin host adicional
    return [h if h.startswith("http") else f"http://{h}" for h in hosts if h]   # normaliza: siempre con http://

OLLAMA, MODELO_OLLAMA = None, os.environ.get("MODELO_OLLAMA")  # ej.: MODELO_OLLAMA="llama3.2:3b" para elegir modelo
for h in _hosts_ollama():   # prueba cada dirección
    try:                                              # /api/tags lista los modelos instalados; timeout corto
        modelos = [m["name"] for m in json.load(urllib.request.urlopen(f"{h}/api/tags", timeout=2))["models"]]
        chat = [m for m in modelos if "embed" not in m]   # descartamos modelos de embeddings (no generan texto)
        if chat:   # hay al menos un modelo de chat
            OLLAMA, MODELO_OLLAMA = h, MODELO_OLLAMA or chat[0]   # ej.: ("http://localhost:11434", "qwen3.6:35b-a3b")
            break   # usamos el primero que responde
    except Exception:                                 # no responde: probamos el siguiente host
        continue   # siguiente dirección

try:                                                  # torch solo hace falta sin Ollama (Colab lo trae instalado)
    import torch                                      # para saber si hay GPU (Colab: Entorno > T4 GPU)
    GPU = torch.cuda.is_available()                   # True si hay GPU disponible
except ImportError:                                   # local sin torch: basta con Ollama
    GPU = False                                       # sin torch asumimos CPU
# Con GPU usamos 1.5B (mejor calidad, ~3 GB); sin GPU, 0.5B (~1 GB) para acotar el tiempo de inferencia en CPU
MODELO_HF = "Qwen/Qwen2.5-1.5B-Instruct" if GPU else "Qwen/Qwen2.5-0.5B-Instruct"
# Nota WSL: para usar el Ollama de Windows desde WSL, inicien Ollama en Windows con OLLAMA_HOST=0.0.0.0
_generador = None                                     # se carga solo la primera vez que se necesita
print("LLM:", f"Ollama {MODELO_OLLAMA} en {OLLAMA}" if OLLAMA else f"Hugging Face {MODELO_HF} (se descarga la 1.ª vez)")

def preguntar_llm(prompt, max_tokens=200):   # misma función en local (Ollama) y en Colab (Hugging Face)
    """Envía un prompt y devuelve el texto. Ej.: preguntar_llm("Resume en una frase: ...") -> "El texto trata de ..." """
    global _generador   # reutiliza el modelo ya cargado
    if OLLAMA:                                        # camino 1: servidor Ollama (rápido, privado, sin internet)
        cuerpo = json.dumps({"model": MODELO_OLLAMA, "prompt": prompt, "stream": False, "think": False,
                             "options": {"temperature": 0, "num_predict": max_tokens}}).encode()  # temperature 0 = repetible
        req = urllib.request.Request(f"{OLLAMA}/api/generate", cuerpo, {"Content-Type": "application/json"})   # petición POST a Ollama
        return json.load(urllib.request.urlopen(req, timeout=600))["response"].strip()   # texto de la respuesta
    if _generador is None:                            # camino 2: transformers (Colab ya lo trae instalado)
        from transformers import pipeline   # carga perezosa: solo si no hay Ollama
        _generador = pipeline("text-generation", model=MODELO_HF,
                              device=0 if GPU else -1)   # 0 = primera GPU, -1 = CPU
    salida = _generador([{"role": "user", "content": prompt}], max_new_tokens=max_tokens, do_sample=False)
    return salida[0]["generated_text"][-1]["content"].strip()   # el último mensaje es la respuesta del modelo

<a id="fase-9"></a>
## Fase 9: Tendencias: BERTopic, LLM en el pipeline y Parquet

a) **BERTopic** sobre los mismos tweets de la fase 3: comparen con LDA el número de temas, sus palabras clave, el % de tweets sin tema (-1) y el cruce con emociones. b) **LLM zero-shot:** clasifiquen 100 tweets de prueba con `preguntar_llm()` (sin entrenar) y compárenlo con su mejor modelo de la fase 4 en F1 macro, tiempo por tweet y respuestas fuera de la lista. c) **Embeddings:** entrenen el mismo clasificador lineal sobre embeddings (`all-MiniLM-L6-v2`) y compárenlo con TF-IDF en macro-F1. d) **Drift:** ordenen por `tweet_id`, entrenen con el tramo más antiguo y midan el PSI de las predicciones en el tramo más reciente. e) Guarden la tabla de resultados en **Parquet**, léanla con Spark y comparen su tamaño con el CSV equivalente. Cierren: ¿en qué escenario se justifica un LLM o un modelo de embeddings frente a TF-IDF + SVM? (costo, privacidad LOPDP, latencia). Citen la lectura de tendencia de la semana (*¿LLM o embeddings? El dilema del costo en la minería de textos*), BERTopic (Grootendorst, 2022) y la documentación de spacy-llm.

> **Pregunta de la guía:** BERTopic vs. LDA; LLM zero-shot y embeddings vs. su mejor modelo (F1 macro, tiempo, costo); PSI de drift; tamaño CSV vs. Parquet.

**Qué analizamos:** Se exploran cuatro tendencias sobre el mismo corpus de tweets: BERTopic frente a LDA, un LLM zero-shot y embeddings frente a TF-IDF + SVM, el drift medido con PSI entre un tramo antiguo y uno reciente (ordenado por tweet_id) y el formato Parquet frente a CSV leído con Spark. Los bloques que necesitan descargar modelos de Hugging Face (BERTopic, LLM, embeddings) están escritos completos pero se saltan si no hay acceso; el drift y Parquet se ejecutan siempre con datos reales.

**Guarda de acceso.** BERTopic y el modelo de embeddings `all-MiniLM-L6-v2` se descargan de Hugging Face, y el LLM puede ser local (Ollama) o un modelo de Hugging Face. Esta celda comprueba (con 5 s de espera) si se puede abrir https://huggingface.co. Si no se puede, cada bloque que lo necesita se salta con un aviso y NO se inventa ningún resultado: queda pendiente de ejecutar en Colab o en local con acceso.

In [ ]:
import os, re, time, json, tempfile, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score

def hay_huggingface():
    """True si https://huggingface.co responde en 5 s (en Colab responde; sin acceso devuelve error 403 o timeout)."""
    try:
        urllib.request.urlopen('https://huggingface.co', timeout=5)
        return True
    except Exception:
        return False

HF_OK = hay_huggingface()
# El LLM puede correr sin Hugging Face si hay Ollama (variable OLLAMA de la celda del modelo de lenguaje)
LLM_OK = bool(OLLAMA) or HF_OK
AVISO = "No ejecutado en este entorno: sin acceso a Hugging Face; ejecutar en Colab"
print("HF_OK =", HF_OK, "| Ollama disponible =", bool(OLLAMA), "| LLM_OK =", LLM_OK)
if not HF_OK:
    print("AVISO:", AVISO, "(BERTopic y embeddings).")
if not LLM_OK:
    print("AVISO:", AVISO, "(LLM zero-shot: tampoco hay Ollama).")

res9 = {'hf_ok': bool(HF_OK), 'llm_ok': bool(LLM_OK), 'bertopic': {'ejecutado': False}, 'llm_zero_shot': {'ejecutado': False},
        'embeddings': {'ejecutado': False}}   # aquí se acumulan las cifras de toda la fase (results/fase9.json)

**a) BERTopic frente a LDA.** BERTopic convierte cada tweet en un embedding, reduce dimensiones (UMAP), agrupa (HDBSCAN) y describe cada grupo con c-TF-IDF (Grootendorst, 2022). A diferencia de LDA, no obliga a elegir K y deja en el tema -1 los tweets que no encajan en ningún grupo. Se usa la misma muestra de 5 000 tweets de la Fase 3 (`muestra_temas`) y `min_topic_size=30`.

In [ ]:
modelo_emb = None
if HF_OK:
    try:
        from sentence_transformers import SentenceTransformer
        modelo_emb = SentenceTransformer('all-MiniLM-L6-v2')   # 384 dimensiones, ~90 MB (se descarga la 1.ª vez)
    except Exception as e:
        print("No se pudo cargar all-MiniLM-L6-v2:", e)
        HF_OK = False
        print("AVISO:", AVISO)

if HF_OK:
    from bertopic import BERTopic
    from umap import UMAP
    from sklearn.feature_extraction.text import CountVectorizer
    textos_temas = muestra_temas['texto_limpio'].tolist()
    t0 = time.time()
    embs_temas = modelo_emb.encode(textos_temas, show_progress_bar=False)   # embeddings de los 5 000 tweets
    umap_fijo = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=SEMILLA)  # semilla -> repetible
    bert = BERTopic(embedding_model=modelo_emb, min_topic_size=30, language='english', umap_model=umap_fijo,
                    vectorizer_model=CountVectorizer(stop_words='english'))   # stopwords fuera para palabras clave legibles
    temas_bert, _ = bert.fit_transform(textos_temas, embeddings=embs_temas)   # tema de cada tweet (-1 = sin tema)
    t_bert = time.time() - t0
    temas_bert = np.array(temas_bert)
    info_bert = bert.get_topic_info()
    print(f"BERTopic ajustado en {t_bert:.1f} s")
    print(info_bert.head(12).to_string(index=False))
else:
    print(AVISO, "-> bloque a) BERTopic saltado.")

**Comparación BERTopic vs LDA.** Se comparan el número de temas, el porcentaje de tweets sin tema, las palabras clave y, para el cruce con emociones, la «pureza» de cada tema: qué fracción de sus tweets comparte la emoción más frecuente del tema (un valor alto indicaría que el tema captura una emoción; se contrasta con el peso de la emoción más común en toda la muestra).

In [ ]:
if HF_OK:
    emo_muestra = muestra_temas['sentiment']
    n_bert = int((info_bert['Topic'] != -1).sum())
    pct_sin_tema = float((temas_bert == -1).mean() * 100)
    # LDA solo asigna tema a los tweets con palabras útiles tras quitar stopwords; los demás quedan sin tema
    pct_lda_sin_tema = round(100 * (1 - len(tema_dom) / len(muestra_temas)), 1)
    claves_bert = {int(t): [w for w, _ in bert.get_topic(int(t))[:8]] for t in info_bert['Topic'] if t != -1}

    def pureza(temas, emociones):
        """Fracción de tweets (con tema) que pertenecen a la emoción mayoritaria de su tema."""
        tabla = pd.crosstab(pd.Series(temas, name='tema'), emociones.reset_index(drop=True).rename('emocion'))
        return float(tabla.max(axis=1).sum() / tabla.values.sum()), tabla

    tema_bert_s = pd.Series(temas_bert, index=muestra_temas.index)
    con_tema = tema_bert_s != -1
    pureza_bert, cruce_bert = pureza(tema_bert_s[con_tema].values, emo_muestra[con_tema])
    pureza_lda, cruce_lda = pureza(tema_dom.values, muestra_temas.loc[tema_dom.index, 'sentiment'])
    base_emocion = float(emo_muestra.value_counts(normalize=True).iloc[0])
    comparacion = pd.DataFrame({'LDA': [K_LDA, pct_lda_sin_tema, round(pureza_lda, 3)],
                                'BERTopic': [n_bert, round(pct_sin_tema, 1), round(pureza_bert, 3)]},
                               index=['n.º de temas', '% de tweets sin tema', 'pureza de emoción'])
    print(comparacion.to_string())
    print(f"Emoción más común en la muestra: {base_emocion:.3f} (referencia para la pureza)")
    print("\nPalabras clave LDA:")
    for i, p in enumerate(palabras_lda):
        print(f"  tema {i}: {', '.join(p[:8])}")
    print("\nPalabras clave BERTopic:")
    for t, p in claves_bert.items():
        print(f"  tema {t}: {', '.join(p)}")
    print("\nCruce BERTopic (tema x emoción, conteos):")
    print(cruce_bert.to_string())
    res9['bertopic'] = {'ejecutado': True, 'n_temas_bertopic': n_bert, 'n_temas_lda': int(K_LDA),
                        'pct_sin_tema_bertopic': round(pct_sin_tema, 2), 'pct_sin_tema_lda': pct_lda_sin_tema,
                        'pureza_bertopic': round(pureza_bert, 4), 'pureza_lda': round(pureza_lda, 4),
                        'palabras_bertopic': {str(k): v for k, v in claves_bert.items()}, 'tiempo_s': round(t_bert, 1)}
else:
    print(AVISO, "-> comparación BERTopic vs LDA pendiente.")

**Qué se analiza:** qué emociones aparecen dentro de cada tema de BERTopic (proporción por fila). Si un tema se concentra en una sola emoción, el tema aporta señal emocional; si se reparte, el tema es temático (cumpleaños, trabajo, música) y no emocional.

In [ ]:
if HF_OK:
    prop = cruce_bert.div(cruce_bert.sum(axis=1), axis=0).loc[cruce_bert.sum(axis=1).sort_values(ascending=False).index[:12]]
    fig, ax = plt.subplots(figsize=(9, 5))
    im = ax.imshow(prop.values, cmap='Purples', aspect='auto')
    ax.set_xticks(range(prop.shape[1])); ax.set_xticklabels(prop.columns, rotation=60, ha='right', fontsize=8)
    ax.set_yticks(range(prop.shape[0])); ax.set_yticklabels([f"{t}: {', '.join(claves_bert[t][:3])}" for t in prop.index], fontsize=8)
    ax.set_title('BERTopic: proporción de cada emoción dentro del tema', color=TINTA, loc='left')
    plt.colorbar(im, ax=ax, label='proporción'); plt.tight_layout()
    plt.savefig('figuras/fase9_bertopic_tema_emocion.png', dpi=150); plt.show()
else:
    print(AVISO, "-> figura BERTopic no generada.")

**b) LLM zero-shot.** Se clasifican los 100 primeros tweets de prueba con `preguntar_llm()` (tal como está definida en el notebook) sin entrenar nada, pidiendo UNA palabra de la lista de emociones. Se mide el F1 macro frente al mejor modelo de la Fase 4 en esos mismos 100 tweets, el tiempo por tweet y el porcentaje de respuestas fuera de la lista. Con solo 100 tweets y 13 clases, el F1 macro es muy ruidoso (varias emociones tienen 0 o 1 ejemplo): se calcula solo sobre las emociones presentes en esos 100.

In [ ]:
etiquetas = list(ETIQUETAS)
X100, y100 = X_test.iloc[:100], y_test.iloc[:100]          # los 100 primeros tweets de prueba
if LLM_OK:
    def limpiar_respuesta(r):
        """Toma la primera palabra de la respuesta, en minúscula y sin signos (el LLM a veces añade puntos o frases)."""
        palabras = re.findall(r"[a-z]+", r.lower())
        return palabras[0] if palabras else ''
    pred_llm, tiempos_llm = [], []
    for texto in X100:
        t0 = time.time()
        crudo = preguntar_llm(f"Clasifica la emoción del tweet con UNA palabra de esta lista {etiquetas}. "
                              f"Tweet: {texto}\nEmoción:", max_tokens=5)
        tiempos_llm.append(time.time() - t0)
        pred_llm.append(limpiar_respuesta(crudo))
    fuera_lista = [p not in etiquetas for p in pred_llm]
    pct_fuera = 100 * float(np.mean(fuera_lista))
    pred_llm_ok = [p if p in etiquetas else 'fuera_de_lista' for p in pred_llm]   # una respuesta inválida cuenta como error
    emociones_100 = sorted(y100.unique())                    # solo las emociones que existen en estos 100
    f1_llm = f1_score(y100, pred_llm_ok, average='macro', labels=emociones_100, zero_division=0)
    t0 = time.time()
    pred_clasico_100 = mejor_modelo.predict(X100)            # mejor modelo de la Fase 4 en los MISMOS 100
    t_clasico_100 = time.time() - t0
    f1_clasico_100 = f1_score(y100, pred_clasico_100, average='macro', labels=emociones_100, zero_division=0)
    t_llm_tweet = float(np.mean(tiempos_llm))
    print(f"LLM zero-shot ({MODELO_OLLAMA if OLLAMA else MODELO_HF}): F1 macro = {f1_llm:.3f} | {t_llm_tweet:.2f} s/tweet | fuera de la lista = {pct_fuera:.0f} %")
    print(f"Mejor modelo Fase 4 ({mejor_nombre}): F1 macro = {f1_clasico_100:.3f} | {1000 * t_clasico_100 / 100:.2f} ms/tweet")
    res9['llm_zero_shot'] = {'ejecutado': True, 'modelo': MODELO_OLLAMA if OLLAMA else MODELO_HF, 'f1_macro_llm': round(f1_llm, 4),
                             'f1_macro_mejor_modelo_mismos_100': round(f1_clasico_100, 4), 's_por_tweet_llm': round(t_llm_tweet, 3),
                             'ms_por_tweet_mejor_modelo': round(1000 * t_clasico_100 / 100, 3), 'pct_fuera_de_lista': round(pct_fuera, 1)}
    resultados = pd.concat([resultados, pd.DataFrame([{'modelo': f'LLM zero-shot (100 tweets, {MODELO_OLLAMA if OLLAMA else MODELO_HF})', 'fase': 9,
                            'f1_macro': round(f1_llm, 4), 'accuracy': round(accuracy_score(y100, pred_llm_ok), 4),
                            'tiempo_s': round(float(np.sum(tiempos_llm)), 2)}])], ignore_index=True)
else:
    print(AVISO, "-> bloque b) LLM zero-shot saltado.")

**Costo aproximado por 1 000 tweets (este bloque sí se ejecuta).** El tamaño del prompt se mide con el texto real; la conversión a tokens usa la regla práctica ≈ 4 caracteres por token (es una aproximación, no el tokenizador de ningún modelo). NO se incluye ninguna tarifa de proveedor: el costo en dinero es tokens × tarifa vigente del proveedor elegido, que debe consultarse. Para el modelo clásico se mide el tiempo real de predicción de 1 000 tweets.

In [ ]:
def prompt_de(texto):
    return (f"Clasifica la emoción del tweet con UNA palabra de esta lista {etiquetas}. Tweet: {texto}\nEmoción:")
chars_prompt = X_test.iloc[:1000].map(lambda t: len(prompt_de(t)))
tokens_entrada_1000 = float(chars_prompt.sum()) / 4          # aproximación: 4 caracteres ≈ 1 token
t0 = time.time()
_ = mejor_modelo.predict(X_test.iloc[:1000])
t_clasico_1000 = time.time() - t0
print(f"Prompt zero-shot: {chars_prompt.mean():.0f} caracteres por tweet (la lista de 13 emociones y la instrucción ocupan ≈ {len(prompt_de('')):.0f} caracteres fijos)")
print(f"1 000 tweets con LLM por API: ≈ {tokens_entrada_1000:,.0f} tokens de entrada (+ ≈ 5 de salida por tweet); costo = tokens x tarifa del proveedor.")
print(f"1 000 tweets con {mejor_nombre} (TF-IDF): {t_clasico_1000:.3f} s de CPU de predicción, sin enviar datos fuera de la organización.")
res9['costo_1000_tweets'] = {'tokens_entrada_aprox': round(tokens_entrada_1000), 'caracteres_prompt_medio': round(float(chars_prompt.mean()), 1),
                             'tiempo_prediccion_mejor_modelo_s': round(t_clasico_1000, 4)}

**c) Embeddings frente a TF-IDF.** Se codifica cada tweet con `all-MiniLM-L6-v2` (vector de 384 dimensiones que captura significado) y se entrena el mismo clasificador lineal (`LinearSVC`) sobre esos vectores. Se elige C en validación y se evalúa en prueba una vez, igual que con TF-IDF. Codificar ~40 000 tweets en CPU tarda varios minutos; con GPU (Colab) es rápido. Si hiciera falta acortar, se puede entrenar con una sub-muestra de `X_train` (variable `N_ENT_EMB`).

In [ ]:
N_ENT_EMB = None                       # None = todo X_train; p. ej. 10000 para una sub-muestra en CPU
if HF_OK:
    X_ent_emb = X_train if N_ENT_EMB is None else X_train.sample(N_ENT_EMB, random_state=SEMILLA)
    y_ent_emb = y_train.loc[X_ent_emb.index]
    t0 = time.time()
    E_train = modelo_emb.encode(X_ent_emb.tolist(), batch_size=128, show_progress_bar=False)
    E_val = modelo_emb.encode(X_val.tolist(), batch_size=128, show_progress_bar=False)
    E_test = modelo_emb.encode(X_test.tolist(), batch_size=128, show_progress_bar=False)
    t_encode = time.time() - t0
    mejor_c, mejor_f1_val = None, -1
    for C in [0.1, 0.3, 1.0]:                                    # C del SVM lineal, elegido en validación
        f1_v = f1_score(y_val, LinearSVC(C=C, random_state=SEMILLA).fit(E_train, y_ent_emb).predict(E_val), average='macro')
        if f1_v > mejor_f1_val:
            mejor_c, mejor_f1_val = C, f1_v
    t0 = time.time()
    clf_emb = LinearSVC(C=mejor_c, random_state=SEMILLA).fit(E_train, y_ent_emb)
    t_fit_emb = time.time() - t0
    pred_emb = clf_emb.predict(E_test)
    f1_emb, acc_emb = f1_score(y_test, pred_emb, average='macro'), accuracy_score(y_test, pred_emb)
    f1_tfidf = f1_score(y_test, pred_test_mejor, average='macro')   # mejor modelo TF-IDF de la Fase 4
    print(f"Embeddings + LinearSVC (C={mejor_c}): F1 macro = {f1_emb:.3f}, accuracy = {acc_emb:.3f} | codificar {t_encode:.0f} s + entrenar {t_fit_emb:.1f} s")
    print(f"TF-IDF + {mejor_nombre} (Fase 4): F1 macro = {f1_tfidf:.3f}")
    res9['embeddings'] = {'ejecutado': True, 'f1_macro_embeddings': round(f1_emb, 4), 'f1_macro_tfidf_mejor': round(f1_tfidf, 4),
                          'accuracy_embeddings': round(acc_emb, 4), 'C': mejor_c, 'n_train_usado': int(len(X_ent_emb)),
                          'tiempo_codificar_s': round(t_encode, 1), 'tiempo_entrenar_s': round(t_fit_emb, 2)}
    resultados = pd.concat([resultados, pd.DataFrame([{'modelo': f'Embeddings MiniLM + LinearSVC (C={mejor_c})', 'fase': 9,
                            'f1_macro': round(f1_emb, 4), 'accuracy': round(acc_emb, 4), 'tiempo_s': round(t_encode + t_fit_emb, 2)}])], ignore_index=True)
else:
    print(AVISO, "-> bloque c) embeddings saltado.")

**d) Drift con PSI.** El drift ocurre cuando los datos de producción dejan de parecerse a los de entrenamiento. Se ordenan los tweets por `tweet_id` (proxy del tiempo: los ids de Twitter crecen con la fecha), se toma el 70 % más antiguo como «tramo antiguo» y el 30 % final como «tramo reciente». Se entrena TF-IDF + LinearSVC con el 80 % del tramo antiguo y se reserva el 20 % restante como referencia (hold-out). El PSI compara la distribución de clases PREDICHAS en la referencia (esperada) con la del tramo reciente (actual): PSI = Σ (a% − e%) · ln(a% / e%), con un epsilon para evitar ceros. Lectura habitual: < 0.1 sin cambio importante; 0.1–0.25 cambio moderado; > 0.25 cambio fuerte (hay que reentrenar o investigar).

In [ ]:
from sklearn.model_selection import train_test_split
ordenado = df.sort_values('tweet_id').reset_index(drop=True)           # orden por tweet_id
corte = int(len(ordenado) * 0.70)
antiguo, reciente = ordenado.iloc[:corte], ordenado.iloc[corte:]       # 70 % antiguo, 30 % reciente
ent_a, ref_a = train_test_split(antiguo, test_size=0.20, stratify=antiguo['sentiment'], random_state=SEMILLA)  # entrenamiento y referencia

def psi(esperado, actual, clases, epsilon=1e-4):
    """PSI entre dos distribuciones de clases. epsilon evita log(0) y divisiones por cero."""
    e = np.array([np.mean(np.array(esperado) == c) for c in clases]) + epsilon
    a = np.array([np.mean(np.array(actual) == c) for c in clases]) + epsilon
    e, a = e / e.sum(), a / a.sum()                                    # vuelven a sumar 1 tras el epsilon
    return float(np.sum((a - e) * np.log(a / e)))

def semaforo(valor):
    return 'sin cambio importante' if valor < 0.1 else ('cambio moderado' if valor < 0.25 else 'cambio fuerte')

t0 = time.time()
modelo_drift = Pipeline([('tfidf', TfidfVectorizer(min_df=2, sublinear_tf=True)),
                         ('clf', LinearSVC(C=0.3, random_state=SEMILLA))]).fit(ent_a['texto_limpio'], ent_a['sentiment'])
t_drift = time.time() - t0
pred_ref = modelo_drift.predict(ref_a['texto_limpio'])
pred_rec = modelo_drift.predict(reciente['texto_limpio'])
psi_pred = psi(pred_ref, pred_rec, ETIQUETAS)
psi_real = psi(ref_a['sentiment'], reciente['sentiment'], ETIQUETAS)   # drift en las etiquetas verdaderas (informativo)
f1_ref = f1_score(ref_a['sentiment'], pred_ref, average='macro')
f1_rec = f1_score(reciente['sentiment'], pred_rec, average='macro')
print(f"tweet_id: antiguo {antiguo['tweet_id'].min()}-{antiguo['tweet_id'].max()} ({len(antiguo)}), reciente {reciente['tweet_id'].min()}-{reciente['tweet_id'].max()} ({len(reciente)})")
print(f"Entrenamiento: {len(ent_a)} tweets | referencia: {len(ref_a)} | reciente: {len(reciente)} | modelo entrenado en {t_drift:.1f} s")
print(f"PSI de las predicciones (referencia vs reciente) = {psi_pred:.4f} -> {semaforo(psi_pred)}")
print(f"PSI de las etiquetas reales (informativo)         = {psi_real:.4f} -> {semaforo(psi_real)}")
print(f"F1 macro: referencia (tramo antiguo) = {f1_ref:.3f} | tramo reciente = {f1_rec:.3f} | diferencia = {f1_rec - f1_ref:+.3f}")

**Qué se analiza:** cómo se reparten las clases predichas por el modelo en la referencia (tramo antiguo) y en el tramo reciente. Barras muy parecidas implican PSI bajo; una barra que cambia mucho (en ACENTO, la emoción con mayor aporte al PSI) indica dónde se movió la distribución.

In [ ]:
dist_ref = pd.Series(pred_ref).value_counts(normalize=True).reindex(ETIQUETAS, fill_value=0) * 100
dist_rec = pd.Series(pred_rec).value_counts(normalize=True).reindex(ETIQUETAS, fill_value=0) * 100
aporte = (dist_rec / 100 - dist_ref / 100) * np.log((dist_rec / 100 + 1e-4) / (dist_ref / 100 + 1e-4))   # aporte de cada clase al PSI
clave = aporte.abs().idxmax()                                           # clase que más aporta
x = np.arange(len(ETIQUETAS)); ancho = 0.4
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.bar(x - ancho / 2, dist_ref.values, ancho, color=GRIS, label='Referencia (tramo antiguo)')
ax.bar(x + ancho / 2, dist_rec.values, ancho, color=[ACENTO if e == clave else TINTA for e in ETIQUETAS], label='Tramo reciente')
ax.set_xticks(x); ax.set_xticklabels(ETIQUETAS, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('% de predicciones', color=TINTA)
ax.set_title(f'Distribución de clases predichas: PSI = {psi_pred:.3f} ({semaforo(psi_pred)}); mayor aporte: {clave}', color=TINTA, loc='left', fontsize=10)
ax.legend(frameon=False, fontsize=9)
ax.spines[['top', 'right']].set_visible(False); ax.grid(False)
plt.tight_layout(); plt.savefig('figuras/fase9_drift_distribuciones.png', dpi=150); plt.show()
res9['drift'] = {'psi_predicciones': round(psi_pred, 4), 'interpretacion': semaforo(psi_pred), 'psi_etiquetas_reales': round(psi_real, 4),
                 'f1_macro_referencia': round(f1_ref, 4), 'f1_macro_reciente': round(f1_rec, 4), 'n_entrenamiento': int(len(ent_a)),
                 'n_referencia': int(len(ref_a)), 'n_reciente': int(len(reciente)), 'clase_mayor_aporte': clave,
                 'tweet_id_corte': int(reciente['tweet_id'].min()),
                 'pct_predicho_referencia': dist_ref.round(1).to_dict(), 'pct_predicho_reciente': dist_rec.round(1).to_dict()}

**e) Parquet y Spark.** Parquet es un formato columnar comprimido: guarda cada columna junta, la comprime mejor y Spark solo lee las columnas que necesita. Se guarda la tabla `resultados` en `data/resultados_emociones.parquet` (pandas + pyarrow), se lee con la sesión de Spark de la Fase 7 y se comparan tamaños con el CSV equivalente. Como la tabla de resultados es diminuta, se repite la comparación con el corpus completo (tweet_id, sentiment, texto_limpio) en CSV frente a Parquet con compresión snappy.

In [ ]:
try:
    spark                                    # la sesión de la Fase 7 sigue activa
except NameError:                            # si se ejecuta esta fase sola
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName('emociones').master('local[*]').config('spark.driver.memory', '4g').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

RUTA_PQ = 'data/resultados_emociones.parquet'
resultados.to_parquet(RUTA_PQ, engine='pyarrow', index=False)          # pandas -> Parquet
print("Tabla de resultados guardada en Parquet:", resultados.shape)
spark.read.parquet(RUTA_PQ).groupBy('modelo').avg('f1_macro').show(truncate=False)   # leída con Spark

def tam_mb(ruta):
    """Tamaño en MB de un archivo o de una carpeta."""
    if os.path.isdir(ruta):
        return sum(os.path.getsize(os.path.join(ruta, f)) for f in os.listdir(ruta)) / 1e6
    return os.path.getsize(ruta) / 1e6

with tempfile.TemporaryDirectory() as tmp:                             # archivos de la comparación: se borran al terminar
    csv_res, csv_cor, pq_cor = f'{tmp}/resultados.csv', f'{tmp}/corpus.csv', f'{tmp}/corpus.parquet'
    resultados.to_csv(csv_res, index=False)
    corpus = df[['tweet_id', 'sentiment', 'texto_limpio']]
    corpus.to_csv(csv_cor, index=False)
    corpus.to_parquet(pq_cor, engine='pyarrow', compression='snappy', index=False)
    tam = {'resultados_csv_mb': tam_mb(csv_res), 'resultados_parquet_mb': tam_mb(RUTA_PQ),
           'corpus_csv_mb': tam_mb(csv_cor), 'corpus_parquet_snappy_mb': tam_mb(pq_cor)}
    # Lectura con Spark del corpus en Parquet: conteo y plan (se ve que solo lee las columnas necesarias)
    sdf_pq = spark.read.parquet(pq_cor)
    n_pq = sdf_pq.count()
    print("Filas leídas por Spark desde el corpus en Parquet:", n_pq, "(igual que df:", n_pq == len(df), ")")
    sdf_pq.groupBy('sentiment').count().explain(mode='formatted')

razon_res = tam['resultados_csv_mb'] / tam['resultados_parquet_mb']
razon_cor = tam['corpus_csv_mb'] / tam['corpus_parquet_snappy_mb']
tabla_tam = pd.DataFrame({'CSV (MB)': [tam['resultados_csv_mb'], tam['corpus_csv_mb']],
                          'Parquet (MB)': [tam['resultados_parquet_mb'], tam['corpus_parquet_snappy_mb']],
                          'razón CSV/Parquet': [razon_res, razon_cor]}, index=['tabla resultados', 'corpus completo (snappy)']).round(6)
print(tabla_tam.to_string())

**Qué se analiza:** el tamaño en disco del mismo dato en CSV y en Parquet. En la tabla diminuta de resultados Parquet no ayuda (los metadatos pesan más que los datos); en el corpus completo, sí.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
etiq = ['tabla resultados', 'corpus completo']
ancho = 0.38; x = np.arange(2)
ax.bar(x - ancho / 2, tabla_tam['CSV (MB)'], ancho, color=GRIS, label='CSV')
ax.bar(x + ancho / 2, tabla_tam['Parquet (MB)'], ancho, color=ACENTO, label='Parquet (snappy)')
for i in range(2):
    ax.text(i - ancho / 2, tabla_tam['CSV (MB)'].iloc[i], f"{tabla_tam['CSV (MB)'].iloc[i]:.3g}", ha='center', va='bottom', fontsize=8, color=TINTA)
    ax.text(i + ancho / 2, tabla_tam['Parquet (MB)'].iloc[i], f"{tabla_tam['Parquet (MB)'].iloc[i]:.3g}", ha='center', va='bottom', fontsize=8, color=TINTA)
ax.set_xticks(x); ax.set_xticklabels(etiq)
ax.set_ylabel('MB en disco', color=TINTA)
ax.set_title('CSV vs Parquet', color=TINTA, loc='left')
ax.legend(frameon=False); ax.spines[['top', 'right']].set_visible(False); ax.grid(False)
plt.tight_layout(); plt.savefig('figuras/fase9_tamano_csv_parquet.png', dpi=150); plt.show()
res9['parquet'] = {k: round(v, 6) for k, v in tam.items()}
res9['parquet'].update({'razon_resultados': round(razon_res, 3), 'razon_corpus': round(razon_cor, 3), 'filas_resultados': int(len(resultados))})
with open('results/fase9.json', 'w', encoding='utf-8') as f:
    json.dump(res9, f, ensure_ascii=False, indent=2, default=str)
print("Guardado results/fase9.json")

**Cierre: ¿cuándo se justifica un LLM o un modelo de embeddings frente a TF-IDF + SVM?** Se resume en cuatro criterios.
- **Costo.** TF-IDF + SVM predice miles de tweets por segundo en una CPU y su costo marginal es casi cero. Un LLM por API cobra por token (el prompt de este ejercicio ya incluye la lista de 13 emociones), y un LLM local necesita GPU o paga latencia. Los embeddings quedan en medio: codificar es caro una vez, pero luego cada texto es un vector reutilizable (búsqueda, agrupamiento, clasificación).
- **Latencia.** El modelo clásico responde en milisegundos; un LLM, en décimas de segundo a segundos por tweet. Para moderación en línea o paneles en tiempo real pesa mucho.
- **Privacidad (LOPDP, Ecuador).** Enviar tweets con datos personales a una API externa es un tratamiento y posible transferencia internacional de datos personales, que exige base legítima, deber de información y garantías con el proveedor. Un modelo clásico o un LLM/embedding local (Ollama, `sentence-transformers` en servidor propio) mantiene los datos dentro de la organización.
- **Mantenimiento.** TF-IDF + SVM se reentrena en segundos y es explicable (palabras con más peso). Un LLM cambia con las versiones del proveedor, requiere vigilar el prompt y las respuestas fuera de la lista, y es más difícil de auditar.
**Regla práctica:** TF-IDF + SVM como línea base y para volumen alto con latencia baja; embeddings cuando importe el significado más allá de las palabras (sinónimos, poco texto etiquetado, búsqueda semántica) y se pueda pagar la codificación; LLM zero-shot cuando no hay datos etiquetados, el volumen es pequeño o se necesita explicar/resumir, y el dato puede salir (o el modelo corre en local).

**Fuentes.** Lectura de tendencia de la semana: *¿LLM o embeddings? El dilema del costo en la minería de textos* (se cita por su título y su idea general: comparar el costo de usar LLM frente a embeddings en tareas de minería de textos; no se reproducen datos de la lectura). Grootendorst, M. (2022). *BERTopic: Neural topic modeling with a class-based TF-IDF procedure*. arXiv:2203.05794. Documentación de spacy-llm (spaCy): integra LLM como componentes de un pipeline de spaCy (tareas como clasificación de texto con prompts), https://spacy.io/usage/large-language-models.

**Conclusión de negocio (Fase 9).**
**Lo que SÍ se ejecutó (datos reales).**
1. **Drift: hay un cambio fuerte.** Ordenando por `tweet_id`, se entrenó TF-IDF + LinearSVC con 22 224 tweets del tramo antiguo (70 % = 27 781) y se midió en 5 557 tweets de referencia y en los 11 907 más recientes. El PSI de las clases predichas es 0.350, por encima del umbral de 0.25 («cambio fuerte»; el de las etiquetas reales, solo informativo, es 0.519). El modelo predice mucho más «worry» (21.2 % → 37.1 %) y «sadness» (6.8 % → 14.8 %) y mucho menos «happiness» (20.0 % → 8.0 %, la clase que más aporta al PSI), «love» (11.2 % → 4.2 %) y «neutral» (36.5 % → 31.3 %). El F1 macro baja de 0.166 en la referencia a 0.142 en el tramo reciente (−0.024). Lectura de negocio: el tono de los tweets cambió con el tiempo (más preocupación y tristeza), así que un modelo entrenado solo con el pasado subestima la alegría; conviene vigilar el PSI de forma periódica y reentrenar cuando supere 0.25. Es un PSI sobre predicciones: indica que cambió el dato o el mundo, no por sí solo que el modelo esté peor (eso lo dice el F1, que sí cayó).
2. **Parquet.** La tabla de resultados de esta corrida de prueba es diminuta (4 filas): en CSV ocupa ≈ 0.29 KB y en Parquet ≈ 3.6 KB (razón CSV/Parquet = 0.08: Parquet pesa MÁS, porque guarda metadatos de esquema). Con el corpus completo (39 688 tweets, tweet_id + sentiment + texto_limpio) el CSV pesa 3.39 MB y el Parquet con snappy 2.25 MB (razón 1.51x, un 34 % menos). Spark lee el Parquet directamente y el plan muestra `ReadSchema` con solo las columnas necesarias. La ganancia de tamaño es moderada porque el dato es sobre todo texto libre, difícil de comprimir; en tablas numéricas o con muchas columnas repetidas la razón suele ser mayor. Por eso Parquet vale por esquema tipado y lectura por columnas más que por el tamaño en este corpus.
**Lo que NO se pudo ejecutar aquí (pendiente de la corrida en Colab o en local con acceso a Hugging Face; este entorno devuelve 403).** El código de a) BERTopic, b) LLM zero-shot y c) embeddings MiniLM está completo y se salta con el aviso «No ejecutado en este entorno: sin acceso a Hugging Face; ejecutar en Colab». NO hay cifras de estos tres bloques; no se reporta ningún F1, tiempo ni % de ruido de ellos.
- Lo siguiente es una EXPECTATIVA cualitativa, no un resultado: BERTopic debería dejar un porcentaje apreciable de tweets en el tema -1 (LDA asigna siempre un tema, 0 % sin tema por construcción) y sus temas deberían ser más temáticos que emocionales (como los de LDA en la Fase 3); un LLM pequeño en zero-shot probablemente dé respuestas fuera de la lista y sea mucho más lento por tweet que el SVM; los embeddings + clasificador lineal podrían igualar o mejorar algo a TF-IDF, pero la etiqueta es ruidosa y está solapada (13 clases), así que no cabe esperar un salto grande. Todo esto debe confirmarse con la corrida real.
- Lo que sí se midió y sirve para el costo: el prompt zero-shot ocupa ≈ 276 caracteres por tweet (≈ 69 000 tokens de entrada por cada 1 000 tweets, con la regla de 4 caracteres por token; es una aproximación) y el SVM predijo 1 000 tweets en ≈ 0.04 s de CPU. El costo en dinero del LLM es tokens × tarifa del proveedor, que no se asume aquí.
**Decisión.** Para este ejercicio (13 clases, corpus pequeño, datos públicos) se mantiene TF-IDF + SVM como modelo de producción por costo, latencia y privacidad, se añade monitoreo de drift con PSI y se completa la comparación con LLM y embeddings en Colab antes de recomendar un cambio.

<a id="fase-final"></a>
## Fase final: aplicación al entorno profesional (individual)

¿Qué textos no estructurados se generan en tu entorno laboral (correos, reclamos, chats, informes, redes sociales) en Ecuador o tu país, y en qué idioma o variante? Describe qué información extraerías (emoción, temas, entidades), qué técnica de esta semana usarías, si necesitarías procesamiento distribuido, y una implicación ética de analizar esos textos.

Cada integrante responde **por separado** (150–250 palabras). Esta sección se califica de forma individual.

### Integrante 1: Kevin Ambuludi

_Respuesta individual_

### Integrante 2: Rodrigo Soria

_Respuesta individual_

### Integrante 3: Christian Ruiz

_Respuesta individual_

## Checklist de cierre — Semana 3

- [ ] Limpié el corpus y documenté cada decisión de limpieza.
- [ ] Analicé POS, NER y dependencias con spaCy usando nlp.pipe.
- [ ] Modelé temas con LDA y los comparé con BERTopic.
- [ ] Comparé Naive Bayes, SVM y una RNN con F1 macro en la misma partición.
- [ ] Repetí el experimento con 3, 5, 7, 9 y 11 clases contra la línea de azar.
- [ ] Repliqué la clasificación en Spark MLlib, consulté con Spark SQL y una función de ventana, e hice un conteo MapReduce con RDD.
- [ ] Comparé un LLM zero-shot y embeddings con mi mejor modelo, medí el drift con PSI y guardé resultados en Parquet.
- [ ] Completé la tabla de alternativas y la reflexión individual.

**Siguiente paso:** Esta semana completa los temas del módulo (datos tabulares, grafos, texto y procesamiento distribuido) y prepara el proyecto integrador con ArXiv: PySpark, spaCy, LDA vs. BERTopic y hoja de ruta MLOps.